# OCR Bilans Fiscaux Algériens — V13c — extraction, contrôles, liasse Excel tracée

> Objectif : 1 PDF → 1 JSON contrôlé → 1 classeur Excel annoté.

## Journal des versions
| Version | Date | Changement |
|---|---|---|
| V13c | 2026-08 | Modèle + recalc résolus depuis `/mnt/Risk/Model` ; écarts d'arrondi ±1 DA annotés d'un commentaire (jaune doux) ; en-tête de chaque feuille = nom du PDF source + page(s) scannée(s) pour traçabilité. |
| V12 | 2026-08 | Version complète précédente (identité, contrôles, transposition). |

## Règle d'or
Ne jamais inventer de valeur : absent/illisible → null.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DEPENDANCES | BILANS_V13
# Objectif: installer le runtime minimal.
# ════════════════════════════════════════════════════════════
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dependances OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | BILANS_V13
# ════════════════════════════════════════════════════════════
import time, json, re, gc, copy, unicodedata, shutil, subprocess, sys
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import openpyxl
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Protection, Side
print('✅ Imports OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | BILANS_V13 — OPTIMISE H100 80GB
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2000 * 32 * 32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 32          # ← H100 : 16 → 32
GPU_BATCH_SIZE = 8               # ← H100 80GB : 4 → 8
PDF_WORKERS = 4                  # ← parallélisme preprocessing PDF
INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v10'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v13.log'
INDEX_PATH = OUTPUT_DIR / 'index_bilans_v13.jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
# ── H100 CUDA tuning ──
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device: ' + DEVICE + ' | PDF: ' + str(len(pdfs)) + ' | JSON: ' + str(JSON_DIR))
print('Batch sizes — classif: ' + str(CLASSIF_BATCH_SIZE) + ' | extraction: ' + str(GPU_BATCH_SIZE) + ' | PDF workers: ' + str(PDF_WORKERS))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — LOG | BILANS_V13
# ════════════════════════════════════════════════════════════
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT MODELE | BILANS_V13 — OPTIMISE H100 (sans flash-attn)
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'

# ── Activer le backend Flash dans SDPA (natif PyTorch >= 2.2, aucun pip) ──
# Sur H100 : PyTorch route automatiquement vers le kernel flash ou mem_efficient
torch.backends.cuda.enable_flash_sdp(True)       # kernel flash (si dispo)
torch.backends.cuda.enable_mem_efficient_sdp(True) # fallback efficace
torch.backends.cuda.enable_math_sdp(False)         # désactiver le fallback lent

try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=False),   # H100 FP8 natif
    attn_implementation='sdpa',                         # ← PyTorch SDPA (pas besoin de flash-attn)
)
model.eval()

# ── torch.compile ──
_COMPILED = False
try:
    model = torch.compile(model, mode='reduce-overhead', fullgraph=False)
    _COMPILED = True
except Exception as e:
    print('⚠️ torch.compile indisponible (fallback normal) : ' + str(e))

# ── Diagnostic : quel backend SDPA est actif ? ──
_sdpa_info = []
if torch.backends.cuda.flash_sdp_enabled(): _sdpa_info.append('flash_sdp')
if torch.backends.cuda.mem_efficient_sdp_enabled(): _sdpa_info.append('mem_efficient_sdp')
print('✅ Modele charge en ' + str(round(time.time()-t0, 1)) + 's'
      + ' | SDPA backends: ' + ', '.join(_sdpa_info)
      + ' | FP8 natif'
      + (' | torch.compile' if _COMPILED else ''))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — UTILITAIRES | BILANS_V13 — OPTIMISE H100
# ════════════════════════════════════════════════════════════
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def strip_accents(s):
    s = str(s)
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def norm_key(s):
    s = strip_accents(str(s)).lower()
    s = re.sub('[^a-z0-9]+', ' ', s)
    return s.strip(' ')

def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img, a

def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

# ── Rendu d'une seule page PDF (pour multiprocessing) ──
def _render_one_page(args):
    pdf_path, page_idx, zoom, max_side = args
    doc = fitz.open(pdf_path)
    matrix = fitz.Matrix(zoom, zoom)
    pix = doc.load_page(page_idx).get_pixmap(matrix=matrix, alpha=False)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()
    img, angle = deskew(img)
    img = resize(img, max_side)
    return {'index': page_idx, 'image': img, 'largeur_px': img.width, 'hauteur_px': img.height, 'rotation_estimee': angle}

def pdf_to_pages(path, zoom=PDF_ZOOM):
    """Rendu PDF parallélisé sur CPU (ne bloque pas le GPU)."""
    doc = fitz.open(path)
    n_pages = len(doc)
    doc.close()
    if n_pages <= 2 or PDF_WORKERS <= 1:
        # Petit PDF : séquentiel (overhead pool > gain)
        doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
        for i in range(len(doc)):
            pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
            img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
            img, angle = deskew(img)
            img = resize(img)
            pages.append({'index': i, 'image': img, 'largeur_px': img.width, 'hauteur_px': img.height, 'rotation_estimee': angle})
        doc.close()
        return pages
    # Gros PDF : parallèle
    args = [(str(path), i, zoom, IMAGE_MAX_SIZE) for i in range(n_pages)]
    with ThreadPoolExecutor(max_workers=PDF_WORKERS) as pool:
        pages = list(pool.map(_render_one_page, args))
    pages.sort(key=lambda p: p['index'])
    return pages

def parse_json(text):
    if not text: return {}
    text = text.strip()
    try: return json.loads(text)
    except Exception: pass
    start = text.find('{'); end = text.rfind('}')
    if start >= 0 and end > start:
        try: return json.loads(text[start:end+1])
        except Exception: return {}
    return {}

def apply_template(messages):
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True, clean_up_tokenization_spaces=False)

# ── Warmup GPU (1 forward pass bidon pour initialiser CUDA graphs / kernels) ──
_WARMED_UP = False
def _warmup_gpu():
    global _WARMED_UP
    if _WARMED_UP or DEVICE != 'cuda': return
    dummy = Image.new('RGB', (256, 256), (255, 255, 255))
    try:
        ask_single('test', dummy)
    except Exception:
        pass
    torch.cuda.empty_cache()
    _WARMED_UP = True
    print('✅ GPU warmup OK')

def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]), 'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images):
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    # ── Preprocessing CPU en thread pendant que le GPU finit le batch précédent ──
    texts = [apply_template(m) for m in msgs]
    inputs = processor(text=texts, images=images, return_tensors='pt', padding=True).to(DEVICE, non_blocking=True)
    t0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len), 'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len, 'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)} for i in range(len(images))]

print('✅ Utilitaires OK (parallélisme PDF + warmup + autocast BF16)')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — SCHEMAS + CLASSIFICATION CONTENU + PROMPTS | BILANS_V13
# (identique V12 : signatures texte, en-tete par page)
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'titre': 'BILAN (ACTIF)', 'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains', 'batiments': 'Batiments', 'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_corporelles': 'Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession', 'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence', 'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises', 'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'immobilisations_financieres': 'Immobilisations financieres', 'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours', 'clients': 'Clients', 'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots et assimiles', 'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
   'creances_emplois_assimiles': 'Creances et emplois assimiles',
   'disponibilites_assimiles': 'Disponibilites et assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants', 'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT', 'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'titre': 'BILAN (PASSIF)', 'cols': ['n', 'n1'], 'postes': {
   'capital_emis': 'Capital emis', 'capital_non_appele': 'Capital non appele', 'primes_reserves': 'Primes et reserves',
   'ecart_reevaluation': 'Ecart de reevaluation', 'ecart_equivalence': 'Ecart d equivalence', 'resultat_net_passif': 'Resultat net',
   'report_a_nouveau': 'Report a nouveau', 'part_societe_consolidante': 'Part de la societe consolidante', 'part_minoritaires': 'Part des minoritaires',
   'total_capitaux_propres': 'TOTAL I', 'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes', 'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits constatés d avance', 'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches', 'impots_passif': 'Impots', 'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif', 'total_passifs_courants': 'TOTAL PASSIFS COURANTS', 'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'titre': 'COMPTE DE RESULTAT', 'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises', 'produits_fabriques': 'Produits Fabriques', 'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux', 'produits_annexes': 'Produits Annexes', 'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net', 'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee', 'subvention_exploitation': 'Subvention d exploitation', 'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues', 'matieres_premieres': 'Matieres premieres', 'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks', 'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services', 'rabais_remises_ristournes_obtenus_achats': 'Rabais remises ristournes obtenus sur achats',
   'autres_consommations': 'Autres consommations',
   'sous_traitance_generale': 'Sous-traitance generale', 'locations': 'Locations', 'entretien_reparations': 'Entretien reparations et maintenance',
   'primes_assurances': 'Primes d assurances', 'personnel_exterieur': 'Personnel exterieur a l entreprise', 'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite', 'deplacements_missions': 'Deplacements missions et receptions', 'rabais_remises_ristournes_obtenus_services': 'Rabais remises ristournes obtenus sur services exterieurs',
   'autres_services': 'Autres services',
   'consommations_exercice': 'II-Consommations de l exercice', 'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
   'charges_personnel': 'Charges de personnel', 'impots_taxes_assimiles': 'Impots et taxes et versements assimiles', 'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels', 'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements', 'provisions': 'Provisions', 'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions', 'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers', 'charges_financieres': 'Charges financieres', 'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire', 'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
   'elements_extraordinaires_charges': 'Elements extraordinaires Charges', 'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats', 'impots_differes_resultats': 'Impots differes sur resultats', 'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'titre': 'DECLARATION IBS', 'cols': ['valeur'], 'kinds': {'nif': 'nif', 'raison_sociale': 'texte', 'activite_principale': 'texte', 'registre_commerce': 'texte', 'adresse_siege': 'texte', 'cac_cabinet': 'texte', 'cac_nom': 'texte', 'exercice_annee': 'annee', 'annee_souscription': 'annee'}, 'postes': {
   'nif': 'Numero d Identification Fiscale', 'raison_sociale': 'Designation de l entreprise', 'activite_principale': 'Activite principale',
   'registre_commerce': 'Registre de Commerce', 'adresse_siege': 'Adresse siege social', 'cac_cabinet': 'Certification des comptes - Cabinet',
   'cac_nom': 'Certification des comptes - Nom CAC', 'exercice_annee': 'Resultat de l exercice - Annee', 'annee_souscription': 'Annee de souscription',
   'chiffre_affaires_global_ht': 'Chiffre d affaires global hors taxes', 'resultat_comptable': 'Resultat comptable', 'resultat_fiscal': 'Resultat fiscal'}},
 'A1': {'titre': '1/ Tableau des mouvements des stocks', 'cols': ['solde_debut', 'debit', 'credit', 'solde_fin'], 'postes': {
   'stocks_marchandises': 'Stocks de marchandises', 'matieres_fournitures': 'Matieres et fournitures', 'autres_approvisionnements': 'Autres approvisionnements',
   'encours_production_biens': 'Encours de production de biens', 'encours_production_services': 'Encours de production de services', 'stocks_produits': 'Stocks de produits',
   'stocks_provenant_immobilisations': 'Stocks provenant d immobilisations', 'stocks_exterieur': 'Stocks a l exterieur', 'total': 'TOTAL'}},
 'A2': {'titre': '2/ Tableau de la fluctuation de la production stockee', 'cols': ['debit', 'credit', 'solde_debiteur', 'solde_crediteur'], 'postes': {
   'production_stockee': 'Production stockee ou destockee (ligne unique si non libellee)'}},
 'A3': {'titre': '3/ Charges de personnel, impots, taxes, autres services', 'cols': ['montant'], 'postes': {
   'charges_locatives': 'Charges locatives et charges de copropriete', 'etudes_recherches': 'Etudes et recherches', 'documentation_divers': 'Documentation et divers',
   'transports_biens': 'Transports de biens et transport collectif du personnel', 'frais_postaux': 'Frais postaux et de telecommunications',
   'services_bancaires': 'Services bancaires et assimiles', 'cotisations_divers': 'Cotisations et divers', 'total_autres_services': 'TOTAL (1)',
   'remunerations_personnel': 'Remunerations du personnel', 'remuneration_exploitant': 'Remunerations de l exploitant individuel (cas d une EURL)',
   'cotisations_sociales': 'Cotisations aux organismes sociaux', 'charges_sociales_exploitant': 'Charges sociales de l exploitant individuel (cas d une EURL)',
   'autres_charges_sociales': 'Autres charges sociales', 'autres_charges_personnel': 'Autres charges de personnels', 'total_charges_personnel': 'TOTAL (2)',
   'impots_sur_remunerations': 'Impots taxes et versements assimiles sur remunerations', 'impots_non_recuperables': 'Impots et taxes non recuperables sur chiffres d affaires',
   'autres_impots_taxes': 'Autres impots et taxes (hors impots sur les resultats)', 'total_impots': 'TOTAL (3)', 'total_general': 'TOTAL (1)+(2)+(3)'}},
 'A4': {'titre': '4/ Autres charges et produits operationnels', 'cols': ['montant'], 'postes': {
   'redevances_concessions_charges': 'Redevances pour concessions brevets licences logiciels (charges)', 'moins_values_sorties_actifs': 'Moins values sur sorties d actifs immobilises non financiers',
   'jetons_presence_charges': 'Jetons de presence (charges)', 'pertes_creances_irrecouvrables': 'Perte sur creances irrecouvrables',
   'quote_part_operations_commun_charges': 'Quote-part de resultat sur operations faites en commun (charges)', 'amendes_penalites_dons': 'Amendes et penalites subventions accordees dons et liberalites',
   'charges_exceptionnelles_gestion': 'Charges exceptionnelles de gestion courante', 'autres_charges_gestion': 'Autres charges de gestion courante', 'total_charges': 'TOTAL (charges)',
   'redevances_concessions_produits': 'Redevances pour concessions brevets licences logiciels (produits)', 'plus_values_sorties_actifs': 'Plus values sur sorties d actifs immobilises non financiers',
   'jetons_presence_produits': 'Jetons de presence et remunerations d administrateurs ou de gerant', 'quotes_parts_subventions_virees': 'Quotes-parts de subventions d investissement virees au resultat',
   'quote_part_operations_commun_produits': 'Quote-part de resultat sur operations faites en commun (produits)', 'rentrees_creances_amorties': 'Rentree sur creances amorties',
   'produits_exceptionnels_gestion': 'Produits exceptionnels sur operations de gestion', 'autres_produits_gestion': 'Autres produits de gestion courante', 'total_produits': 'TOTAL (produits)'}},
 'A5': {'titre': '5/ Tableau des amortissements et pertes de valeurs', 'cols': ['dotations_cumulees_debut', 'dotations_exercice', 'diminutions_elements_sortis', 'dotations_cumulees_fin', 'dotations_fiscales_exercice', 'ecarts'], 'postes': {
   'goodwill': 'Goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles', 'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations', 'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants', 'total': 'TOTAL'}},
 'A6': {'titre': '6/ Tableau des immobilisations creees ou acquises', 'cols': ['montants_bruts', 'tva_deduite', 'montant_net_a_amortir'], 'postes': {
   'goodwill': 'Goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles', 'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations', 'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants', 'total': 'TOTAL'}},
 'A7': {'titre': '7/ Tableau des immobilisations cedees', 'dynamic': True, 'str_cols': ['date_acquisition'], 'cols': ['date_acquisition', 'montant_net_actif', 'amortissements_pratiques', 'valeur_nette_comptable', 'prix_cession', 'plus_value', 'moins_value'], 'postes': {}},
 'A8': {'titre': '8/ Tableau des provisions et pertes de valeurs', 'cols': ['provisions_cumulees_debut', 'dotations_exercice', 'reprises_exercice', 'provisions_cumulees_fin'], 'postes': {
   'pertes_valeur_stocks': 'Pertes de valeurs sur stocks', 'pertes_valeur_creances': 'Pertes de valeurs sur creances', 'pertes_valeur_actions': 'Pertes de valeurs sur actions et parts sociales',
   'provisions_pensions': 'Provisions pour pensions et obligations similaires', 'provisions_litiges': 'Provisions sur litiges', 'autres_provisions_personnel': 'Autres provisions liees au personnel',
   'provisions_impots': 'Provisions pour impots', 'autres_provisions': 'Autres provisions', 'total': 'TOTAL'}},
 'A81': {'titre': '8/1 Releve des pertes de valeurs sur creances', 'dynamic': True, 'cols': ['valeur_creance', 'perte_valeur_constituee'], 'postes': {}},
 'A82': {'titre': '8/2 Releve des pertes de valeurs sur actions', 'dynamic': True, 'cols': ['valeur_nominale_debut', 'perte_valeur_constituee', 'valeur_nette_comptable'], 'postes': {}},
 'A9': {'titre': '9/ Tableau de determination du resultat fiscal', 'cols': ['montant'], 'postes': {
   'resultat_net_benefice': 'I. Resultat net de l exercice Benefice', 'resultat_net_perte': 'I. Resultat net de l exercice Perte',
   'charges_immeubles_non_affectes': 'Charges des immeubles non affectes directement a l exploitation', 'quote_part_cadeaux_publicitaires': 'Quote-part des cadeaux publicitaires non deductibles',
   'quote_part_sponsoring': 'Quote-part du sponsoring et parrainage non deductibles', 'frais_reception': 'Frais de reception non deductibles', 'cotisations_dons': 'Cotisations et dons non deductibles',
   'impots_taxes_non_deductibles': 'Impots et taxes non deductibles', 'provisions_non_deductibles': 'Provisions non deductibles', 'amortissements_non_deductibles': 'Amortissements non deductibles',
   'quote_part_frais_rd': 'Quote-part des frais de recherche developpement non deductibles', 'amortissements_credit_bail_preneur': 'Amortissements non deductibles credit bail (Preneur)',
   'loyers_hors_produits_financiers_bailleur': 'Loyers hors produits financiers (bailleur)', 'ibs_impot_exigible': 'Impots sur les benefices - Impot exigible sur le resultat',
   'ibs_impot_differe': 'Impots sur les benefices - Impot differe (variation)', 'pertes_valeur_non_deductibles': 'Pertes de valeurs non deductibles', 'amendes_penalites': 'Amendes et penalites',
   'autres_reintegrations': 'Autres reintegrations', 'total_reintegrations': 'Total des reintegrations', 'plus_values_cession_actif_immobilise': 'Plus values sur cession d elements d actif immobilises',
   'produits_plus_values_actions_bourse': 'Produits et plus values de cession des actions et titres assimiles cotes en bourse', 'revenus_distribution_benefices': 'Revenus provenant de la distribution des benefices',
   'amortissements_credit_bail_bailleur': 'Amortissements credit bail (Bailleur)', 'loyers_hors_charges_financieres_preneur': 'Loyers hors charges financieres (Preneur)',
   'complement_amortissements': 'Complement d amortissements', 'autres_deductions': 'Autres deductions', 'total_deductions': 'Total des deductions',
   'total_deficits_a_deduire': 'Total des deficits a deduire', 'resultat_fiscal_benefice': 'Resultat fiscal (I+II-III-IV) Benefice', 'resultat_fiscal_deficit': 'Resultat fiscal (I+II-III-IV) Deficit'}},
 'A10': {'titre': '10/ Tableau d affectation du resultat et des reserves (N-1)', 'cols': ['montant'], 'postes': {
   'origine_report_a_nouveau_n1': 'Report a nouveau de l exercice N-1', 'origine_resultat_n1': 'Resultat de l exercice N-1', 'origine_prelevements_reserves': 'Prelevements sur reserves',
   'origine_total': 'TOTAL (origine)', 'affectation_reserves': 'Reserves', 'affectation_augmentation_capital': 'Augmentation du capital', 'affectation_dividendes': 'Dividendes',
   'affectation_report_a_nouveau': 'Report a nouveau', 'affectation_total': 'TOTAL (affectation)'}},
 'A11': {'titre': '11/ Tableau des participations', 'dynamic': True, 'cols': ['capitaux_propres', 'dont_capital', 'quote_part_capital_pct', 'resultat_dernier_exercice', 'prets_avances', 'dividendes_encaisses', 'valeur_comptable_titres'], 'postes': {}},
 'A12': {'titre': '12/ Commissions courtages redevances honoraires sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'adresse'], 'cols': ['nif', 'adresse', 'montant_percu'], 'postes': {}},
 'A13': {'titre': '13/ Taxe sur l activite professionnelle', 'dynamic': True, 'cols': ['ca_imposable', 'ca_exonere', 'tap_acquittee'], 'postes': {}},
 'ST': {'titre': 'Operations de sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'article', 'adresse', 'reference_contrat'], 'cols': ['nif', 'article', 'adresse', 'reference_contrat', 'montant'], 'postes': {}},
 'REM': {'titre': 'Remunerations versees aux membres de certaines societes', 'dynamic': True, 'str_cols': ['annee_versement'], 'cols': ['nombre_parts', 'annee_versement', 'traitement_emoluments', 'representation_mission_forfait', 'representation_mission_remboursement', 'frais_pro_forfait', 'frais_pro_remboursement'], 'postes': {}},
 'DIST': {'titre': 'Repartition des produits des actions et parts sociales distribues', 'cols': ['montant'], 'postes': {
   'montant_global_brut': 'Montant global brut des distributions', 'paye_par_societe': 'Paye par la societe elle meme',
   'paye_par_etablissement': 'Paye par un etablissement charge du service des titres', 'total_revenus_repartis': 'Montant total des revenus repartis'}}
}
ANNEXE_NUM_MAP = {1:'A1', 2:'A2', 3:'A3', 4:'A4', 5:'A5', 6:'A6', 7:'A7', 8:'A8', 9:'A9', 10:'A10', 11:'A11', 12:'A12', 13:'A13'}

TITLE_SIGNATURES = [
    ('A81', ['RELEVE DES PERTES DE VALEURS SUR CREANC'], []),
    ('A82', ['RELEVE DES PERTES DE VALEURS SUR ACTIONS'], []),
    ('A1',  ['MOUVEMENTS DES STOCKS'], []),
    ('A2',  ['FLUCTUATION DE LA PRODUCTION STOCKEE'], []),
    ('A3',  ['CHARGES DE PERSONNEL', 'VERSEMENTS ASSIMILES'], []),
    ('A4',  ['AUTRES CHARGES ET PRODUITS OPERATIONNELS'], []),
    ('A5',  ['AMORTISSEMENTS ET PERTES DE VALEURS'], []),
    ('A6',  ['IMMOBILISATIONS CREEES OU ACQUISES'], []),
    ('A7',  ['IMMOBILISATIONS CEDEES'], []),
    ('A8',  ['PROVISIONS ET PERTES DE VALEURS'], ['RELEVE DES PERTES']),
    ('A9',  ['DETERMINATION DU RESULTAT FISCAL'], []),
    ('A10', ['AFFECTATION DU RESULTAT ET DES RESERVES'], []),
    ('A11', ['TABLEAU DES PARTICIPATIONS'], []),
    ('A12', ['COMMISSIONS ET COURTAGES'], []),
    ('A13', ['TAXE SUR L ACTIVITE PROFESSIONNELLE'], []),
    ('ST',  ['OPERATIONS DE SOUS-TRAITANCE'], ['COMMISSIONS ET COURTAGES']),
    ('REM', ['REMUNERATIONS VERSEES AUX MEMBRES'], []),
    ('DIST',['REPARTITION DES PRODUITS DES ACTIONS'], []),
]
# Signatures TCR suite (2e page) : priment sur les annexes car ces
# libelles n'apparaissent que dans le TCR, jamais dans les annexes.
TCR_SUITE_SIGNATURES = [
    ('TCR', ['RESULTAT OPERATIONNEL'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT FINANCIER'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT NET DE L EXERCICE'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT ORDINAIRE'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['EXCEDENT BRUT'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['VALEUR AJOUTEE'], ['DETERMINATION DU RESULTAT FISCAL']),
]

MAIN_SIGNATURES = [
    ('ACTIF',  ['BILAN', 'ACTIF'],  ['PASSIF']),
    ('PASSIF', ['BILAN', 'PASSIF'], []),
    ('TCR',    ['COMPTE DE RESULTAT'], []),
    ('DECL',   ['DECLARATION'], []),
]
def _norm(t):
    t = strip_accents(t or '').upper()
    t = t.replace(chr(39), ' ').replace(chr(8217), ' ')
    t = re.sub(r'[^A-Z0-9/ ]+', ' ', t)
    return ' '.join(t.split())
_TITLE_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs]) for c, ms, fs in TITLE_SIGNATURES]
_TCR_SUITE_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs])
                   for c, ms, fs in TCR_SUITE_SIGNATURES]
_MAIN_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs]) for c, ms, fs in MAIN_SIGNATURES]
def _match_signatures(t, signatures):
    found = []
    for code, musts, forbids in signatures:
        if all(m in t for m in musts) and not any(f in t for f in forbids):
            if code not in found: found.append(code)
    return found
def _num_from_segment(t):
    codes, nums = [], []
    for m in re.finditer(r'(?:^| )(1[0-3]|[0-9])/([0-9])?(?![0-9])', t):
        n = int(m.group(1)); s = m.group(2)
        code = ('A8' + s) if (n == 8 and s in ('1', '2')) else ANNEXE_NUM_MAP.get(n)
        if code and code not in codes:
            codes.append(code); nums.append(n)
    return codes, nums
def types_from_title(titre):
    brut = _norm(titre)
    if not brut: return [], []
    segments = [_norm(s) for s in re.split(r'\s\|\s', titre or '') if s.strip()]
    segments = [s for s in segments if s] or [brut]
    types, nums = [], []
    for seg in segments:
        # TCR suite : soldes intermediaires propres au TCR (2e page)
        par_tcr_suite = _match_signatures(seg, _TCR_SUITE_SIGS)
        par_texte = _match_signatures(seg, _TITLE_SIGS)
        if par_tcr_suite:
            for c in par_tcr_suite:
                if c not in types: types.append(c)
        elif par_texte:
            for c in par_texte:
                if c not in types: types.append(c)
        else:
            codes, ns = _num_from_segment(seg)
            for c in codes:
                if c not in types: types.append(c)
            nums.extend(ns)
    if ('A81' in types or 'A82' in types) and 'A8' in types:
        if not any('PROVISIONS ET PERTES DE VALEURS' in s for s in segments): types.remove('A8')
    if not types:
        types = _match_signatures(brut, _MAIN_SIGS)
        if 'ACTIF' in types and 'PASSIF' in types:
            types = ['ACTIF'] if brut.find('ACTIF') < brut.find('PASSIF') else ['PASSIF']
    return types, nums

PROMPT_CLASSIF = chr(10).join([
 'Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).',
 'Ta seule tache : recopier les TITRES DE TABLEAUX imprimes sur cette page.',
 'QU EST-CE QU UN TITRE : au-dessus d un tableau, souvent cadre/souligne, commence par 1/ 2/ ... 13/ 8/1 8/2, ou titre central BILAN (ACTIF)/(PASSIF), COMPTE DE RESULTAT, DECLARATION...',
 'CE QUI N EST PAS UN TITRE : libelles de lignes, en-tetes de colonnes, mentions N.I.F/Designation/Exercice, IMPRIME DESTINE A L ADMINISTRATION.',
 'ATTENTION : 9/ contient une ligne interne I. Resultat net de l exercice (Compte de resultat) : c est un LIBELLE, pas un titre.',
 'Le COMPTE DE RESULTAT (TCR) occupe souvent DEUX pages : la 1re porte le titre COMPTE DE RESULTAT, la 2e continue avec V-Resultat operationnel et se termine par IX-Resultat net. Les deux pages sont de type TCR, pas ANNEXE.',
 'UNE PAGE PEUT CONTENIR DEUX TABLEAUX : regarde HAUT et BAS. Recopie TOUS les titres, meme coupes.',
 'RECOPIE LE LIBELLE EN ENTIER, mot pour mot, avec son numero.',
 'Titres de reference : 1/ mouvements des stocks ; 2/ fluctuation production stockee ; 3/ charges personnel impots taxes autres services ; 4/ autres charges et produits operationnels ; 5/ amortissements et pertes de valeurs ; 6/ immobilisations creees ou acquises ; 7/ immobilisations cedees ; 8/ provisions et pertes de valeurs ; 8/1 releve pertes creances ; 8/2 releve pertes actions ; 9/ determination du resultat fiscal ; 10/ affectation resultat et reserves (N-1) ; 11/ participations ; 12/ commissions courtages redevances honoraires sous-traitance ; 13/ taxe activite professionnelle.',
 'CAS PARTICULIER — PAGE SANS TITRE DE TABLEAU :',
 'Si la page ne porte AUCUN titre (ni numero N/ ni titre central encadre),',
 'identifie le tableau par son CONTENU :',
 '',
 '  SUITE DU TCR (type TCR) :',
 '  La page contient des lignes en gras numerotees en chiffres romains :',
 '    V-Resultat operationnel',
 '    VI-Resultat financier',
 '    VII-Resultat ordinaire (V+VI)',
 '    VIII-Resultat extraordinaire',
 '    IX-RESULTAT NET DE L EXERCICE',
 '  et des lignes de detail comme : Autres produits operationnels,',
 '  Dotations aux amortissements, Produits financiers, Charges financieres,',
 '  Impots exigibles sur resultats, Impots differes sur resultats.',
 '  Les en-tetes de colonnes (Debit/Credit) ne sont PAS repetes sur cette page',
 '  (ils etaient sur la page precedente du TCR).',
 '  -> Repondre: {"titre": "COMPTE DE RESULTAT (suite)", "type": "TCR"}',
 '',
 '  TABLEAU 9 (type ANNEXE) — A NE PAS CONFONDRE :',
 '  Le tableau 9 se reconnait par :',
 '  - un titre encadre : 9/ Tableau de determination du resultat fiscal',
 '  - une structure en 4 sections : I. Resultat net (Benefice/Perte),',
 '    II. Reintegrations (liste de charges non deductibles),',
 '    III. Deductions, IV. Deficits anterieurs a deduire',
 '  - une seule colonne de montants',
 '  - se termine par Resultat fiscal (I+II-III-IV) Benefice / Deficit',
 '  -> C est une ANNEXE, pas la suite du TCR.',
 '',
 'Reponds en JSON strict: {"titre": "<titres separes par |>", "type": "<ACTIF|PASSIF|TCR|DECL|ANNEXE|AUTRE>"}'])

RULES = [
 'REGLES: JSON valide uniquement, sans markdown, sans backticks.',
 'Aucune valeur inventee. Case vide ou absent: null. Illisible: null.',
 'Montants en nombres JSON sans separateurs de milliers.',
 'Montant entre parentheses = negatif: (1 553 799) devient -1553799.',
 'Textes (nif, adresses, noms, dates) en chaines telles quelles.']

def build_prompt(types):
    L = []
    L.append('Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).')
    L.append('Extrais en JSON strict les tableaux suivants, identifies par leur code.')
    L.append('Structure: { type_page, numero_page_imprimee, titre_page, entete: {entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite, serie_g}, puis une cle par code de tableau present.')
    # Le TCR peut occuper 2 pages : la 2e n a pas de titre mais le meme schema
    if 'TCR' in types:
        L.append('Le COMPTE DE RESULTAT peut occuper 2 pages. Si cette page est '
                 'la suite du TCR (pas de titre, colonnes DEBIT/CREDIT, lignes '
                 'Resultat operationnel a Resultat net), extrais les postes '
                 'normalement avec le code TCR.')
    L.append('EN-TETE — obligatoire sur CHAQUE page :')
    L.append('- nif : 15 chiffres colles, sans barre ni espace; recopie ce qui est lisible.')
    L.append('- entreprise : texte apres Designation de l entreprise, forme juridique comprise.')
    L.append('- exercice : date apres Exercice clos le (jj/mm/aaaa); si Exercice du..au.., remplis exercice_du/exercice_au et date de fin dans exercice.')
    for t in types:
        spec = SCHEMAS[t]
        L.append('--- Code ' + t + ' : ' + spec['titre'] + ' ---')
        if spec.get('dynamic'):
            L.append('Tableau a lignes libres. Retourne: {lignes: [ {libelle_imprime, valeurs: {colonne: nombre ou texte ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            L.append('Seulement les lignes non vides; garde les totaux si presents.')
        else:
            L.append('Retourne: {lignes: [ {row_code, libelle_imprime, valeurs: {colonne: nombre ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            if t == 'DECL': L.append('Les valeurs peuvent etre des nombres ou des textes selon le poste.')
            L.append('Row_code autorises:')
            for k, lab in spec['postes'].items(): L.append('- ' + k + ' : ' + lab)
            L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L.extend(RULES)
    return chr(10).join(L)
VALID_CODES = set(SCHEMAS.keys()) | {'AUTRE', 'BLANCHE'}
print('✅ Schemas + classification + en-tete (V13) OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — NORMALISATION | BILANS_V13
# ════════════════════════════════════════════════════════════
def norm_str(v):
    if v is None: return None
    if isinstance(v, bool): return None
    s = ' '.join(str(v).split())
    return s if s and s.lower() not in ('null','none','n/a','na','-') else None
def norm_montant(v):
    if v is None: return None
    if isinstance(v, bool): return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    if s.lower() in ('null','none','n/a','na','-'): return None
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif ',' in s: s = s.replace(',', '')
    elif s.count('.') > 1: s = s.replace('.', '')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_nif(v):
    s = norm_str(v)
    return re.sub('[^0-9]', '', s) if s else None
def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall('20[0-9]{2}', s)
    return m[-1] if m else None
def norm_by_kind(v, kind):
    if kind == 'texte': return norm_str(v)
    if kind == 'nif': return norm_nif(v)
    if kind == 'annee': return norm_annee4(v)
    return norm_montant(v)
def normalise_fixed(t, block):
    spec = SCHEMAS[t]; kinds = spec.get('kinds', {})
    lignes = (block or {}).get('lignes') or []
    label_to_key = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for lg in lignes:
        if not isinstance(lg, dict): continue
        rc = norm_str(lg.get('row_code'))
        if rc not in spec['postes']:
            lab = norm_str(lg.get('libelle_imprime'))
            if lab and norm_key(lab) in label_to_key: rc = label_to_key[norm_key(lab)]
        if rc in spec['postes']: row_map[rc] = lg
    out = {}
    for key in spec['postes']:
        vals = (row_map.get(key) or {}).get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        out[key] = {}
        for col in spec['cols']:
            out[key][col] = norm_by_kind(vals.get(col), kinds.get(key, 'montant'))
    return out
def normalise_dynamic(t, block):
    spec = SCHEMAS[t]; str_cols = set(spec.get('str_cols', []))
    out = []
    for lg in (block or {}).get('lignes') or []:
        if not isinstance(lg, dict): continue
        vals = lg.get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        row = {'libelle_imprime': norm_str(lg.get('libelle_imprime'))}
        for col in spec['cols']:
            row[col] = norm_str(vals.get(col)) if col in str_cols else norm_montant(vals.get(col))
        if row['libelle_imprime'] or any(v is not None for k, v in row.items() if k != 'libelle_imprime'): out.append(row)
    return out
def normalise_table(t, block):
    if t == 'AUTRE': return block or {}
    if SCHEMAS[t].get('dynamic'): return normalise_dynamic(t, block)
    return normalise_fixed(t, block)
def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict): ent = {}
    return {'entreprise': norm_str(ent.get('entreprise')), 'nif': norm_nif(ent.get('nif')), 'exercice': norm_str(ent.get('exercice')),
            'exercice_du': norm_str(ent.get('exercice_du')), 'exercice_au': norm_str(ent.get('exercice_au')), 'adresse': norm_str(ent.get('adresse')),
            'activite': norm_str(ent.get('activite')), 'serie_g': norm_str(ent.get('serie_g'))}
def count_values(obj):
    count = 0
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'brut': continue
            count += count_values(v)
    elif isinstance(obj, list):
        for v in obj: count += count_values(v)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool): count += 1
    elif isinstance(obj, str) and obj: count += 1
    return count
print('✅ Normalisation OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — CONSTRUCTION PAGES JSON | BILANS_V13
# ════════════════════════════════════════════════════════════
def build_blank_page(page, fichier_source):
    n = page['index'] + 1
    return {'page_id': 'p' + str(n).zfill(3), 'pdf_page_index': page['index'], 'numero_page_scannee': n, 'numero_page_imprimee': None, 'fichier_source': fichier_source,
            'classification': {'type_page': 'BLANCHE', 'types': [], 'annexe_numeros': [], 'sous_type_page': 'PAGE_VIERGE', 'titre_page': None, 'page_annexe': False, 'page_utile': False, 'page_blanche': True},
            'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
            'statut_extraction': {'statut': 'BLANCHE', 'nb_tableaux': 0, 'nb_champs_extraits': 0, 'commentaire': 'Page blanche detectee, non envoyee au VLM.'},
            'entete_page': {}, 'donnees': {}, 'tokens_in': 0, 'tokens_out': 0, 'temps_s': 0.0}
def build_page_object(page, types, data, rep, fichier_source):
    n = page['index'] + 1
    data = data if isinstance(data, dict) else {}
    donnees = {'brut': data}
    for t in types:
        if t == 'AUTRE': donnees['AUTRE'] = {'tables': data.get('tables') or []}
        else: donnees[t] = normalise_table(t, data.get(t))
    statut = 'OK' if data else 'ECHEC_EXTRACTION'
    nums = page.get('annexe_nums') or []
    return {'page_id': 'p' + str(n).zfill(3), 'pdf_page_index': page['index'], 'numero_page_scannee': n,
            'numero_page_imprimee': data.get('numero_page_imprimee') if isinstance(data.get('numero_page_imprimee'), int) else None,
            'fichier_source': fichier_source,
            'classification': {'type_page': types[0] if len(types) == 1 else (types[0] if types else 'AUTRE'), 'types': types, 'annexe_numeros': nums,
                               'sous_type_page': ' '.join(types) if types else 'AUTRE', 'titre_page': norm_str(data.get('titre_page')),
                               'page_annexe': any(t.startswith('A') or t in ('ST','REM','DIST') for t in types), 'page_utile': bool(types), 'page_blanche': False},
            'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
            'statut_extraction': {'statut': statut, 'nb_tableaux': len(types), 'nb_champs_extraits': count_values(donnees), 'commentaire': None},
            'entete_page': normalise_entete(data), 'donnees': donnees,
            'tokens_in': rep.get('tokens_in') if rep else 0, 'tokens_out': rep.get('tokens_out') if rep else 0, 'temps_s': rep.get('elapsed') if rep else 0.0}
def merge_core(base, new):
    if new is None: return base
    if base is None: return copy.deepcopy(new)
    for key, cols in new.items():
        if key not in base: base[key] = copy.deepcopy(cols)
        elif isinstance(cols, dict):
            for col, val in cols.items():
                if base[key].get(col) is None and val is not None: base[key][col] = val
    return base
def build_synthese(page_objects):
    synthese = {'actif': None, 'passif': None, 'tcr': None, 'decl': None, 'annexes': {}, 'annexes_pages': []}
    for page in page_objects:
        types = page['classification'].get('types') or []
        donnees = page.get('donnees', {})
        if 'ACTIF' in types and synthese['actif'] is None: synthese['actif'] = donnees.get('ACTIF')
        if 'PASSIF' in types and synthese['passif'] is None: synthese['passif'] = donnees.get('PASSIF')
        if 'TCR' in types: synthese['tcr'] = merge_core(synthese['tcr'], donnees.get('TCR'))
        if 'DECL' in types and synthese['decl'] is None: synthese['decl'] = donnees.get('DECL')
        for t in types:
            if t in ('ACTIF','PASSIF','TCR','DECL','AUTRE'): continue
            if synthese['annexes'].get(t) is None: synthese['annexes'][t] = donnees.get(t)
        if any(t not in ('ACTIF','PASSIF','TCR','DECL','AUTRE') for t in types):
            synthese['annexes_pages'].append({'page_id': page['page_id'], 'numero_page_scannee': page['numero_page_scannee'], 'types': types, 'annexe_numeros': page['classification'].get('annexe_numeros')})
    return synthese
def build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed):
    nb_bl = sum(1 for p in page_objects if p['classification']['type_page'] == 'BLANCHE')
    nb_ut = sum(1 for p in page_objects if p['classification']['type_page'] != 'BLANCHE')
    nb_an = sum(1 for p in page_objects if p['classification'].get('page_annexe'))
    return {'schema_version': '13.0', 'type_document': 'liasse_fiscale_algerienne_serie_g',
            'document': {'document_id': 'doc_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + pdf_path.stem, 'fichier_source': pdf_path.name,
                         'nb_pages_pdf': len(pages), 'nb_pages_scannes': len(pages), 'nb_pages_blanches': nb_bl, 'nb_pages_utiles': nb_ut, 'nb_pages_annexes': nb_an,
                         'date_extraction': datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'duree_extraction_s': round(elapsed, 2),
                         'modele_extraction': MODEL_PATH.split('/')[-1], 'prompt_version': 'v13', 'referentiel_version': 'liasse_serie_g_v13',
                         'tokens_in': tok_in, 'tokens_out': tok_out, 'tokens_total': tok_in + tok_out},
            'pages': page_objects, 'synthese': build_synthese(page_objects)}
print('✅ Construction pages JSON OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — PIPELINE PRINCIPAL | BILANS_V13
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: ' + str(len(a_traiter)) + ' | deja traites: ' + str(len(deja)))
t_total = time.time(); n_ok = 0; n_err = 0; index_records = []; prompt_cache = {}
# ── Warmup GPU avant le premier PDF ──
_warmup_gpu()

for num, pdf_path in enumerate(a_traiter, start=1):
    prefix = '[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] '
    t_pdf = time.time()
    log(prefix + 'DEMARRAGE ' + pdf_path.name)
        torch.cuda.empty_cache()  # ← defragmentation entre PDFs
    try:
        t0 = time.time()
        pages = pdf_to_pages(pdf_path)
        log(prefix + 'rendu termine: ' + str(len(pages)) + ' pages en ' + str(round(time.time()-t0, 1)) + 's')
        page_objects = []; active_pages = []
        for p in pages:
            if is_blank(p['image']): page_objects.append(build_blank_page(p, pdf_path.name))
            else: active_pages.append(p)
        log(prefix + 'pages blanches: ' + str(len(page_objects)) + ' | pages actives: ' + str(len(active_pages)))
        tok_in = 0; tok_out = 0
        if active_pages:
            t0 = time.time()
            mini = [resize(p['image'], 600) for p in active_pages]
            reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE): reps1 += ask_batch(PROMPT_CLASSIF, bs)
            tok_in += sum(r['tokens_in'] for r in reps1); tok_out += sum(r['tokens_out'] for r in reps1)
            log(prefix + 'classification terminee en ' + str(round(time.time()-t0, 1)) + 's')
            pages_typed = []; type_counts = {}
            for p, rep in zip(active_pages, reps1):
                d = parse_json(rep['text'])
                titre = norm_str(d.get('titre'))
                t_model = (norm_str(d.get('type')) or '').upper()
                types, nums = types_from_title(titre)
                if not types:
                    if t_model in ('ACTIF','PASSIF','TCR','DECL'): types = [t_model]
                    else: types = ['AUTRE']
                p['types'] = types; p['annexe_nums'] = nums
                pages_typed.append((p, types))
                label = '+'.join(types); type_counts[label] = type_counts.get(label, 0) + 1
            log(prefix + 'types (contenu): ' + ', '.join(str(k) + '=' + str(v) for k, v in sorted(type_counts.items())))
            del mini, reps1; gc.collect()
            groups = {}
            for p, types in pages_typed: groups.setdefault(tuple(types), []).append(p)
            extracted = {}
            for key_types, plist in groups.items():
                prompt = prompt_cache.get(key_types)
                if prompt is None:
                    prompt = build_prompt(list(key_types)) if key_types != ('AUTRE',) else ('Lis cette page d un dossier fiscal algerien. Extrais en JSON strict: type_page, titre_page, entete, tables (liste de {table_libelle, colonnes, lignes}). ' + chr(10).join(RULES))
                    prompt_cache[key_types] = prompt
                nb_batches = (len(plist) + GPU_BATCH_SIZE - 1) // GPU_BATCH_SIZE
                batch_no = 0
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    batch_no += 1
                    page_nums = [str(p['index'] + 1) for p in batch]
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + '/' + str(nb_batches) + ' pages [' + ','.join(page_nums) + ']')
                    t0 = time.time()
                    reps = ask_batch(prompt, [p['image'] for p in batch])
                    for p, rep in zip(batch, reps):
                        tok_in += rep['tokens_in']; tok_out += rep['tokens_out']
                        extracted[p['index']] = build_page_object(p, list(key_types), parse_json(rep['text']), rep, pdf_path.name)
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + ' terminee en ' + str(round(time.time()-t0, 1)) + 's | tokens=' + str(sum(r['tokens_in'] + r['tokens_out'] for r in reps)))
                    gc.collect(); torch.cuda.empty_cache()
            for p in active_pages:
                if p['index'] in extracted: page_objects.append(extracted[p['index']])
                else: page_objects.append(build_page_object(p, ['AUTRE'], {}, None, pdf_path.name))
        page_objects.sort(key=lambda x: x['pdf_page_index'])
        elapsed = time.time() - t_pdf
        result = build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed)
        json_path = JSON_DIR / (pdf_path.stem + '.json')
        with open(json_path, 'w', encoding='utf-8') as f: json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        n_ok += 1
        index_records.append({'fichier': pdf_path.name, 'json_path': str(json_path), 'nb_pages': len(pages),
                              'nb_pages_utiles': result['document']['nb_pages_utiles'], 'nb_pages_blanches': result['document']['nb_pages_blanches'],
                              'nb_pages_annexes': result['document']['nb_pages_annexes'], 'tokens_total': tok_in + tok_out,
                              'duree_s': round(elapsed, 2), 'date_extraction': result['document']['date_extraction']})
        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        log(prefix + 'OK ' + pdf_path.name + ' | ' + str(round(elapsed, 1)) + 's | tok=' + str(tok_in + tok_out) + ' | pages=' + str(len(pages)) + ' | utiles=' + str(result['document']['nb_pages_utiles']) + ' | blanches=' + str(result['document']['nb_pages_blanches']) + ' | annexes=' + str(result['document']['nb_pages_annexes']) + ' | ETA ' + str(round(eta/3600, 2)) + 'h')
    except Exception as e:
        n_err += 1
        import traceback as _tb
        log(prefix + 'ERREUR ' + pdf_path.name + ' — ' + type(e).__name__ + ': ' + str(e))
        with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(_tb.format_exc() + chr(10))
        continue
with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    for rec in index_records: f.write(json.dumps(rec, ensure_ascii=False) + chr(10))
log('✅ Termine en ' + str(round(time.time() - t_total, 1)) + 's | OK ' + str(n_ok) + ' | Erreurs ' + str(n_err))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 11 — VALIDATION RAPIDE | BILANS_V13
# ════════════════════════════════════════════════════════════
files = sorted(JSON_DIR.glob('*.json'))
if not files: print('Aucun JSON pour le moment.')
else:
    d = json.load(open(files[-1], encoding='utf-8'))
    print('Fichier:', d['document']['fichier_source'])
    print('Pages:', d['document']['nb_pages_pdf'], '| utiles:', d['document']['nb_pages_utiles'], '| blanches:', d['document']['nb_pages_blanches'], '| annexes:', d['document']['nb_pages_annexes'])
    for p in d['pages']:
        c = p['classification']
        print(p['page_id'], '| scan', p['numero_page_scannee'], '|', '+'.join(c.get('types') or [c['type_page']]), '| nums', c.get('annexe_numeros'), '| champs:', p['statut_extraction']['nb_champs_extraits'])
    syn = d['synthese']
    print('Synthese: actif', syn['actif'] is not None, '| passif', syn['passif'] is not None, '| tcr', syn['tcr'] is not None, '| decl', syn['decl'] is not None)
    print('Annexes presentes:', ', '.join(sorted(k for k, v in syn['annexes'].items() if v is not None)) or 'aucune')

---
# Partie 2 — Contrôles et transposition (V13)

| Cellule | Rôle |
|---|---|
| 12 | Identité du dossier (NIF 15, millésimes, concordance pages) |
| 13 | Règles : formules internes + contrôles croisés |
| 14 | Moteur de cohérence (recalcul, comparaison, enrichissement JSON) |
| 15 | Mapping vers le classeur Excel |
| 16 | Transposition V13 : modèle depuis /mnt/Risk/Model, commentaires d'arrondi ±1, traçabilité PDF+page par feuille |
| 17 | Exécution sur tous les dossiers |
| 18 | Tests hors modèle |

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 12 — IDENTITE DU DOSSIER | BILANS_V13 (identique V12)
# ════════════════════════════════════════════════════════════
def nif_15(valeur):
    if valeur is None: return None, 'ABSENT'
    chiffres = re.sub(r'\D', '', str(valeur))
    if not chiffres: return None, 'ABSENT'
    if len(chiffres) == 15: return chiffres, 'CONFORME'
    if len(chiffres) < 15: return chiffres.zfill(15), 'COMPLETE'
    return chiffres[:15], 'TRONQUE'
def cle_nom(valeur):
    if not valeur: return None
    s = ''.join(c for c in unicodedata.normalize('NFD', str(valeur)) if unicodedata.category(c) != 'Mn').upper()
    s = re.sub(r'\b(SARL|EURL|SPA|SNC|ETS|STE|SOCIETE|GROUPE)\b', ' ', s)
    s = re.sub(r'[^A-Z0-9]+', ' ', s)
    return ' '.join(s.split()) or None
def annee_exercice(entete):
    for cle in ('exercice', 'exercice_au', 'exercice_du'):
        v = entete.get(cle)
        if not v: continue
        annees = re.findall(r'(19|20)\d{2}', str(v))
        if annees: return int(annees[-1])
    return None
def date_cloture(entete):
    for cle in ('exercice', 'exercice_au'):
        v = entete.get(cle)
        if v and re.search(r'\d{2}/\d{2}/\d{4}', str(v)): return re.search(r'\d{2}/\d{2}/\d{4}', str(v)).group()
    return norm_str(entete.get('exercice')) if entete.get('exercice') else None
def _majoritaire(valeurs):
    vals = [v for v in valeurs if v not in (None, '')]
    if not vals: return None, 0, 0
    compte = Counter(vals); val, n = compte.most_common(1)[0]
    return val, n, len(vals)
def construire_identite(doc):
    par_page, alertes = [], []
    for page in doc.get('pages', []):
        if page['classification'].get('page_blanche'): continue
        ent = page.get('entete_page') or {}
        n15, statut_nif = nif_15(ent.get('nif'))
        par_page.append({'page_id': page['page_id'], 'numero_page_scannee': page['numero_page_scannee'], 'types': page['classification'].get('types') or [],
                         'entreprise': norm_str(ent.get('entreprise')), 'entreprise_cle': cle_nom(ent.get('entreprise')), 'nif_brut': norm_str(ent.get('nif')),
                         'nif_15': n15, 'nif_statut': statut_nif, 'exercice_clos': date_cloture(ent), 'annee_n': annee_exercice(ent)})
    nif_ret, n_nif, t_nif = _majoritaire([p['nif_15'] for p in par_page])
    nom_ret, n_nom, t_nom = _majoritaire([p['entreprise_cle'] for p in par_page])
    an_ret, n_an, t_an = _majoritaire([p['annee_n'] for p in par_page])
    libelle = None
    for p in par_page:
        if p['entreprise_cle'] == nom_ret and p['entreprise']:
            if libelle is None or len(p['entreprise']) > len(libelle): libelle = p['entreprise']
    cloture, _, _ = _majoritaire([p['exercice_clos'] for p in par_page])
    def divergences(champ, retenu):
        return [{'page_id': p['page_id'], 'numero_page_scannee': p['numero_page_scannee'], 'valeur': p[champ]} for p in par_page if p[champ] not in (None, '') and p[champ] != retenu]
    for champ, retenu, libelle_champ, gravite in [('nif_15', nif_ret, 'NIF', 'critique'), ('entreprise_cle', nom_ret, 'raison sociale', 'critique'), ('annee_n', an_ret, 'exercice clos', 'majeur')]:
        div = divergences(champ, retenu)
        if div: alertes.append({'code': 'IDENTITE_DIVERGENTE', 'champ': libelle_champ, 'gravite': gravite, 'valeur_retenue': retenu, 'pages_divergentes': div,
                                'message': str(len(div)) + ' page(s) portent un(e) ' + libelle_champ + ' different(e) de la valeur majoritaire.'})
    couverture = {}
    for champ, libelle_champ in [('nif_15', 'NIF'), ('entreprise', 'raison sociale'), ('annee_n', 'exercice clos')]:
        absentes = [p['page_id'] for p in par_page if not p[champ]]
        couverture[champ] = {'pages_renseignees': len(par_page) - len(absentes), 'pages_totales': len(par_page), 'pages_absentes': absentes}
        if absentes: alertes.append({'code': 'ENTETE_INCOMPLETE', 'champ': libelle_champ, 'gravite': 'mineur', 'pages_sans_valeur': absentes,
                                     'message': str(len(absentes)) + ' page(s) sur ' + str(len(par_page)) + ' sans ' + libelle_champ + ' lisible.'})
    orphelines = [p['page_id'] for p in par_page if not p['nif_15'] and not p['entreprise'] and not p['annee_n']]
    if orphelines: alertes.append({'code': 'PAGE_ORPHELINE', 'champ': 'en-tete', 'gravite': 'majeur', 'pages_sans_valeur': orphelines,
                                   'message': str(len(orphelines)) + ' page(s) sans aucun element d identite.'})
    non_conformes = [{'page_id': p['page_id'], 'statut': p['nif_statut'], 'nif_brut': p['nif_brut']} for p in par_page if p['nif_statut'] in ('COMPLETE', 'TRONQUE')]
    if non_conformes: alertes.append({'code': 'NIF_NON_CONFORME', 'champ': 'NIF', 'gravite': 'mineur', 'pages': non_conformes, 'message': 'NIF non conforme 15 positions sur certaines pages.'})
    doc['identite'] = {'entreprise': libelle, 'entreprise_cle': nom_ret, 'nif_15': nif_ret, 'exercice_clos': cloture, 'annee_n': an_ret,
                       'annee_n1': (an_ret - 1) if isinstance(an_ret, int) else None, 'libelle_n': ('N : ' + str(an_ret)) if an_ret else 'N',
                       'libelle_n1': ('N-1 : ' + str(an_ret - 1)) if an_ret else 'N-1',
                       'concordance': {'nif': {'pages_concordantes': n_nif, 'pages_renseignees': t_nif}, 'entreprise': {'pages_concordantes': n_nom, 'pages_renseignees': t_nom}, 'exercice': {'pages_concordantes': n_an, 'pages_renseignees': t_an}},
                       'couverture_entete': couverture, 'dossier_homogene': not any(a['gravite'] == 'critique' for a in alertes), 'pages': par_page, 'alertes': alertes}
    return doc['identite']
print('✅ Identite dossier OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 13 — REGLES DE CALCUL | BILANS_V13 (identique V12)
# ════════════════════════════════════════════════════════════
COLS_ACTIF = ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1']
COLS_PASSIF = ['n', 'n1']
FORMULES = [
 # ── Sous-totaux intermediaires ACTIF ──
 ('ACTIF', 'immobilisations_corporelles', COLS_ACTIF, [('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles')]),
 ('ACTIF', 'immobilisations_financieres', COLS_ACTIF, [('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF', 'creances_emplois_assimiles', COLS_ACTIF, [('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles')]),
 ('ACTIF', 'disponibilites_assimiles', COLS_ACTIF, [('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 # ── Totaux principaux ──
 ('ACTIF', 'total_actif_non_courant', COLS_ACTIF, [('+','ecarts_acquisition_goodwill'),('+','immobilisations_incorporelles'),('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles'),('+','immobilisations_en_concession'),('+','immobilisations_en_cours'),('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF', 'total_actif_courant', COLS_ACTIF, [('+','stocks_encours'),('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles'),('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 ('ACTIF', 'total_general_actif', COLS_ACTIF, [('+','total_actif_non_courant'),('+','total_actif_courant')]),
 ('PASSIF', 'total_capitaux_propres', COLS_PASSIF, [('+','capital_emis'),('-','capital_non_appele'),('+','primes_reserves'),('+','ecart_reevaluation'),('+','ecart_equivalence'),('+','resultat_net_passif'),('+','report_a_nouveau'),('+','part_societe_consolidante'),('+','part_minoritaires')]),
 ('PASSIF', 'total_passifs_non_courants', COLS_PASSIF, [('+','emprunts_dettes_financieres'),('+','impots_differes_provisionnes'),('+','autres_dettes_non_courantes'),('+','provisions_produits_avance')]),
 ('PASSIF', 'total_passifs_courants', COLS_PASSIF, [('+','fournisseurs_rattaches'),('+','impots_passif'),('+','autres_dettes'),('+','tresorerie_passif')]),
 ('PASSIF', 'total_general_passif', COLS_PASSIF, [('+','total_capitaux_propres'),('+','total_passifs_non_courants'),('+','total_passifs_courants')]),
 ('TCR', 'chiffre_affaires_net', 'NET', [('+','ventes_marchandises'),('+','produits_fabriques'),('+','prestations_services'),('+','ventes_travaux'),('+','produits_annexes'),('+','rabais_remises_ristournes_accordes')]),
 ('TCR', 'production_exercice', 'NET', [('+','chiffre_affaires_net'),('+','production_stockee_destockee'),('+','production_immobilisee'),('+','subvention_exploitation')]),
 ('TCR', 'consommations_exercice', 'NET', [('+','achats_marchandises_vendues'),('+','matieres_premieres'),('+','autres_approvisionnements'),('+','variation_stocks'),('+','achats_etudes_prestations'),('+','autres_consommations'),('+','rabais_remises_ristournes_obtenus_achats'),('+','sous_traitance_generale'),('+','locations'),('+','entretien_reparations'),('+','primes_assurances'),('+','personnel_exterieur'),('+','remuneration_intermediaires'),('+','publicite'),('+','deplacements_missions'),('+','rabais_remises_ristournes_obtenus_services'),('+','autres_services')]),
 ('TCR', 'valeur_ajoutee_exploitation', 'NET', [('+','production_exercice'),('+','consommations_exercice')]),
 ('TCR', 'excedent_brut_exploitation', 'NET', [('+','valeur_ajoutee_exploitation'),('+','charges_personnel'),('+','impots_taxes_assimiles')]),
 ('TCR', 'resultat_operationnel', 'NET', [('+','excedent_brut_exploitation'),('+','autres_produits_operationnels'),('+','autres_charges_operationnelles'),('+','dotations_amortissements'),('+','provisions'),('+','pertes_valeur'),('+','reprises_pertes_valeur_provisions')]),
 ('TCR', 'resultat_financier', 'NET', [('+','produits_financiers'),('+','charges_financieres')]),
 ('TCR', 'resultat_ordinaire', 'NET', [('+','resultat_operationnel'),('+','resultat_financier')]),
 ('TCR', 'resultat_extraordinaire', 'NET', [('+','elements_extraordinaires_produits'),('+','elements_extraordinaires_charges')]),
 ('TCR', 'resultat_net_exercice', 'NET', [('+','resultat_ordinaire'),('+','resultat_extraordinaire'),('+','impots_exigibles_resultats'),('+','impots_differes_resultats')]),
 ('A1', 'total', ['solde_debut','debit','credit','solde_fin'], [('+','stocks_marchandises'),('+','matieres_fournitures'),('+','autres_approvisionnements'),('+','encours_production_biens'),('+','encours_production_services'),('+','stocks_produits'),('+','stocks_provenant_immobilisations'),('+','stocks_exterieur')]),
 ('A3', 'total_autres_services', ['montant'], [('+','charges_locatives'),('+','etudes_recherches'),('+','documentation_divers'),('+','transports_biens'),('+','frais_postaux'),('+','services_bancaires'),('+','cotisations_divers')]),
 ('A3', 'total_charges_personnel', ['montant'], [('+','remunerations_personnel'),('+','remuneration_exploitant'),('+','cotisations_sociales'),('+','charges_sociales_exploitant'),('+','autres_charges_sociales'),('+','autres_charges_personnel')]),
 ('A3', 'total_impots', ['montant'], [('+','impots_sur_remunerations'),('+','impots_non_recuperables'),('+','autres_impots_taxes')]),
 ('A3', 'total_general', ['montant'], [('+','total_autres_services'),('+','total_charges_personnel'),('+','total_impots')]),
 ('A4', 'total_charges', ['montant'], [('+','redevances_concessions_charges'),('+','moins_values_sorties_actifs'),('+','jetons_presence_charges'),('+','pertes_creances_irrecouvrables'),('+','quote_part_operations_commun_charges'),('+','amendes_penalites_dons'),('+','charges_exceptionnelles_gestion'),('+','autres_charges_gestion')]),
 ('A4', 'total_produits', ['montant'], [('+','redevances_concessions_produits'),('+','plus_values_sorties_actifs'),('+','jetons_presence_produits'),('+','quotes_parts_subventions_virees'),('+','quote_part_operations_commun_produits'),('+','rentrees_creances_amorties'),('+','produits_exceptionnels_gestion'),('+','autres_produits_gestion')]),
 ('A5', 'total', ['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'], [('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A6', 'total', ['montants_bruts','tva_deduite','montant_net_a_amortir'], [('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A8', 'total', ['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'], [('+','pertes_valeur_stocks'),('+','pertes_valeur_creances'),('+','pertes_valeur_actions'),('+','provisions_pensions'),('+','provisions_litiges'),('+','autres_provisions_personnel'),('+','provisions_impots'),('+','autres_provisions')]),
 ('A9', 'total_reintegrations', ['montant'], [('+','charges_immeubles_non_affectes'),('+','quote_part_cadeaux_publicitaires'),('+','quote_part_sponsoring'),('+','frais_reception'),('+','cotisations_dons'),('+','impots_taxes_non_deductibles'),('+','provisions_non_deductibles'),('+','amortissements_non_deductibles'),('+','quote_part_frais_rd'),('+','amortissements_credit_bail_preneur'),('+','loyers_hors_produits_financiers_bailleur'),('+','ibs_impot_exigible'),('+','ibs_impot_differe'),('+','pertes_valeur_non_deductibles'),('+','amendes_penalites'),('+','autres_reintegrations')]),
 ('A9', 'total_deductions', ['montant'], [('+','plus_values_cession_actif_immobilise'),('+','produits_plus_values_actions_bourse'),('+','revenus_distribution_benefices'),('+','amortissements_credit_bail_bailleur'),('+','loyers_hors_charges_financieres_preneur'),('+','complement_amortissements'),('+','autres_deductions')]),
 ('A10', 'origine_total', ['montant'], [('+','origine_report_a_nouveau_n1'),('+','origine_resultat_n1'),('+','origine_prelevements_reserves')]),
 ('A10', 'affectation_total', ['montant'], [('+','affectation_reserves'),('+','affectation_augmentation_capital'),('+','affectation_dividendes'),('+','affectation_report_a_nouveau')]),
]
FORMULES_LIGNE = [
 ('ACTIF', 'net_n', [('+','montant_brut'),('-','amortissements_provisions_pertes')]),
 ('A1', 'solde_fin', [('+','solde_debut'),('+','debit'),('-','credit')]),
 ('A5', 'dotations_cumulees_fin', [('+','dotations_cumulees_debut'),('+','dotations_exercice'),('-','diminutions_elements_sortis')]),
 ('A5', 'ecarts', [('+','dotations_exercice'),('-','dotations_fiscales_exercice')]),
 ('A7', 'valeur_nette_comptable', [('+','montant_net_actif'),('-','amortissements_pratiques')]),
 ('A8', 'provisions_cumulees_fin', [('+','provisions_cumulees_debut'),('+','dotations_exercice'),('-','reprises_exercice')]),
 ('A82', 'valeur_nette_comptable', [('+','valeur_nominale_debut'),('-','perte_valeur_constituee')]),
]
CONTROLES = [
 ('Equilibre du bilan (N)', 'critique', [('+','ACTIF','total_general_actif','net_n')], [('+','PASSIF','total_general_passif','n')]),
 ('Equilibre du bilan (N-1)', 'critique', [('+','ACTIF','total_general_actif','net_n1')], [('+','PASSIF','total_general_passif','n1')]),
 ('Resultat net : TCR = bilan passif (N)', 'critique', [('+','TCR','resultat_net_exercice','NET')], [('+','PASSIF','resultat_net_passif','n')]),
 ('Resultat net : TCR = bilan passif (N-1)', 'critique', [('+','TCR','resultat_net_exercice','NET_N1')], [('+','PASSIF','resultat_net_passif','n1')]),
 ('Resultat net : TCR = ligne I du tableau 9', 'critique', [('+','TCR','resultat_net_exercice','NET')], [('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant')]),
 ('Stocks : total A1 (fin) = stocks bilan (N)', 'majeur', [('+','A1','total','solde_fin')], [('+','ACTIF','stocks_encours','net_n')]),
 ('Stocks : total A1 (debut) = stocks bilan (N-1)', 'majeur', [('+','A1','total','solde_debut')], [('+','ACTIF','stocks_encours','net_n1')]),
 ('Production stockee : solde A2 = ligne TCR', 'majeur', [('+','A2','production_stockee','solde_debiteur'),('-','A2','production_stockee','solde_crediteur')], [('+','TCR','production_stockee_destockee','n_debit'),('-','TCR','production_stockee_destockee','n_credit')]),
 ('Amortissements : cumul fin A5 = colonne amort. bilan', 'majeur', [('+','A5','total','dotations_cumulees_fin')], [('+','ACTIF','total_actif_non_courant','amortissements_provisions_pertes')]),
 ('Amortissements : dotations A5 = dotations TCR', 'majeur', [('+','A5','total','dotations_exercice')], [('+','TCR','dotations_amortissements','n_debit')]),
 ('Charges A3 : TOTAL(1) = autres services TCR', 'majeur', [('+','A3','total_autres_services','montant')], [('+','TCR','autres_services','n_debit')]),
 ('Charges A3 : TOTAL(2) = charges personnel TCR', 'majeur', [('+','A3','total_charges_personnel','montant')], [('+','TCR','charges_personnel','n_debit')]),
 ('Charges A3 : TOTAL(3) = impots et taxes TCR', 'majeur', [('+','A3','total_impots','montant')], [('+','TCR','impots_taxes_assimiles','n_debit')]),
 ('A4 : total charges = autres charges TCR', 'majeur', [('+','A4','total_charges','montant')], [('+','TCR','autres_charges_operationnelles','n_debit')]),
 ('A4 : total produits = autres produits TCR', 'majeur', [('+','A4','total_produits','montant')], [('+','TCR','autres_produits_operationnels','n_credit')]),
 ('Cessions : plus-values A7 = tableau 4', 'majeur', [('+','A7',' TOTAL ','plus_value')], [('+','A4','plus_values_sorties_actifs','montant')]),
 ('Cessions : moins-values A7 = tableau 4', 'majeur', [('+','A7',' TOTAL ','moins_value')], [('+','A4','moins_values_sorties_actifs','montant')]),
 ('Provisions : dotations A8 = provisions+pertes TCR', 'majeur', [('+','A8','total','dotations_exercice')], [('+','TCR','provisions','n_debit'),('+','TCR','pertes_valeur','n_debit')]),
 ('Provisions : reprises A8 = reprises TCR', 'majeur', [('+','A8','total','reprises_exercice')], [('+','TCR','reprises_pertes_valeur_provisions','n_credit')]),
 ('Provisions : 8/1 = pertes creances tableau 8', 'majeur', [('+','A81',' TOTAL ','perte_valeur_constituee')], [('+','A8','pertes_valeur_creances','provisions_cumulees_fin')]),
 ('Provisions : 8/2 = pertes actions tableau 8', 'majeur', [('+','A82',' TOTAL ','perte_valeur_constituee')], [('+','A8','pertes_valeur_actions','provisions_cumulees_fin')]),
 ('Fiscal : IBS exigible = impots exigibles TCR', 'majeur', [('+','A9','ibs_impot_exigible','montant')], [('+','TCR','impots_exigibles_resultats','n_debit')]),
 ('Fiscal : resultat = I + reint - ded - deficits', 'critique', [('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')], [('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant'),('+','A9','total_reintegrations','montant'),('-','A9','total_deductions','montant'),('-','A9','total_deficits_a_deduire','montant')]),
 ('Fiscal : resultat fiscal DECL = tableau 9', 'majeur', [('+','DECL','resultat_fiscal','valeur')], [('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')]),
 ('Fiscal : resultat comptable DECL = RN TCR', 'majeur', [('+','DECL','resultat_comptable','valeur')], [('+','TCR','resultat_net_exercice','NET')]),
 ('Fiscal : CA global DECL = CA net TCR', 'majeur', [('+','DECL','chiffre_affaires_global_ht','valeur')], [('+','TCR','chiffre_affaires_net','NET')]),
 ('Affectation : origine = affectation', 'critique', [('+','A10','origine_total','montant')], [('+','A10','affectation_total','montant')]),
 ('Affectation : resultat N-1 = RN bilan (N-1)', 'majeur', [('+','A10','origine_resultat_n1','montant')], [('+','PASSIF','resultat_net_passif','n1')]),
 ('Affectation : reserves = variation primes/reserves', 'majeur', [('+','A10','affectation_reserves','montant')], [('+','PASSIF','primes_reserves','n'),('-','PASSIF','primes_reserves','n1')]),
 ('Affectation : augm. capital = variation capital', 'majeur', [('+','A10','affectation_augmentation_capital','montant')], [('+','PASSIF','capital_emis','n'),('-','PASSIF','capital_emis','n1')]),
 ('Tableau 12 = remuneration intermediaires TCR', 'majeur', [('+','A12',' TOTAL ','montant_percu')], [('+','TCR','remuneration_intermediaires','n_debit')]),
 ('Distributions : global = societe + etablissement', 'majeur', [('+','DIST','montant_global_brut','montant')], [('+','DIST','paye_par_societe','montant'),('+','DIST','paye_par_etablissement','montant')]),
 ("Amortissements : ecart A5 = reintegration au tableau 9", 'indicatif',
  [('+','A5','total','ecarts')], [('+','A9','amortissements_non_deductibles','montant')]),
 ("Cessions : plus-values A7 = deduction au tableau 9", 'indicatif',
  [('+','A7','__TOTAL__','plus_value')], [('+','A9','plus_values_cession_actif_immobilise','montant')]),
 ("Cessions : sorties A7 = diminutions du tableau 5", 'indicatif',
  [('+','A7','__TOTAL__','amortissements_pratiques')], [('+','A5','total','diminutions_elements_sortis')]),
 ("Fiscal : IBS differe reintegre = impots differes du TCR", 'majeur',
  [('+','A9','ibs_impot_differe','montant')], [('+','TCR','impots_differes_resultats','n_debit')]),
 ("Affectation : report a nouveau = report du bilan (N)", 'indicatif',
  [('+','A10','affectation_report_a_nouveau','montant')], [('+','PASSIF','report_a_nouveau','n')]),
 ("Affectation : dividendes = distributions du tableau DIST", 'indicatif',
  [('+','A10','affectation_dividendes','montant')], [('+','DIST','montant_global_brut','montant')]),
 ("Sous-traitance : tableau ST = sous-traitance generale du TCR", 'indicatif',
  [('+','ST','__TOTAL__','montant')], [('+','TCR','sous_traitance_generale','n_debit')]),
 ("TAP : CA impose + exonere = chiffre d affaires net du TCR", 'indicatif',
  [('+','A13','__TOTAL__','ca_imposable'),('+','A13','__TOTAL__','ca_exonere')],
  [('+','TCR','chiffre_affaires_net','NET')]),
 ("Participations : valeur des titres = poste du bilan", 'indicatif',
  [('+','A11','__TOTAL__','valeur_comptable_titres')], [('+','ACTIF','autres_participations_creances','net_n')]),
]
print('✅ Regles OK — ' + str(len(FORMULES)) + ' formules, ' + str(len(FORMULES_LIGNE)) + ' formules ligne, ' + str(len(CONTROLES)) + ' controles')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 14 — MOTEUR DE COHERENCE | BILANS_V13 (identique V12)
# ════════════════════════════════════════════════════════════
TOLERANCE_DA = 1.0
def lire_bloc(bloc):
    if isinstance(bloc, list):
        sortie = []
        for i, row in enumerate(bloc):
            if not isinstance(row, dict): continue
            vals = {k: v for k, v in row.items() if k != 'libelle_imprime'}
            sortie.append(('ligne_' + str(i).zfill(3), vals, row.get('libelle_imprime')))
        return sortie
    if isinstance(bloc, dict):
        if isinstance(bloc.get('lignes'), list):
            sortie = []
            for i, lg in enumerate(bloc['lignes']):
                if not isinstance(lg, dict): continue
                cle = lg.get('row_code') or ('ligne_' + str(i).zfill(3))
                sortie.append((cle, lg.get('valeurs') or {}, lg.get('libelle_imprime')))
            return sortie
        return [(rc, vals, None) for rc, vals in bloc.items() if isinstance(vals, dict)]
    return []
def iter_blocs(doc):
    for page in doc.get('pages', []):
        for tab, bloc in (page.get('donnees') or {}).items():
            if tab in ('brut', 'AUTRE') or bloc in (None, {}, []): continue
            yield page, tab, bloc
def _index_postes(doc):
    idx, dyn = {}, {}
    for _page, tab, bloc in iter_blocs(doc):
        for cle, vals, _lib in lire_bloc(bloc):
            renseignees = {k: v for k, v in (vals or {}).items() if v is not None}
            if renseignees: idx.setdefault((tab, cle), {}).update(renseignees)
            if cle != 'total':
                for col, v in renseignees.items():
                    if isinstance(v, (int, float)) and not isinstance(v, bool): dyn[(tab, col)] = dyn.get((tab, col), 0.0) + float(v)
    for (tab, col), somme in dyn.items(): idx.setdefault((tab, ' TOTAL '), {})[col] = somme
    return idx
def _valeur(idx, tab, rc, col):
    vals = idx.get((tab, rc))
    if vals is None: return None
    if col in ('NET', 'NET_N1'):
        suffixe = ('n_credit', 'n_debit') if col == 'NET' else ('n1_credit', 'n1_debit')
        c, d = vals.get(suffixe[0]), vals.get(suffixe[1])
        if c is None and d is None: return None
        return float(c or 0) - float(d or 0)
    v = vals.get(col)
    if isinstance(v, bool) or not isinstance(v, (int, float)): return None
    return float(v)
def _somme(idx, termes):
    total, vus = 0.0, 0
    for signe, tab, rc, col in termes:
        v = _valeur(idx, tab, rc, col)
        if v is not None:
            total += v if signe == '+' else -v; vus += 1
    return total if vus else None
def _statut(ecart, tol):
    if abs(ecart) <= 1e-9: return 'coherent'
    if abs(ecart) <= tol: return 'coherent_arrondi'
    return 'ecart_significatif'
def verifier_coherence(doc, tolerance=TOLERANCE_DA):
    idx = _index_postes(doc)
    resultats_formules, resultats_controles = [], []
    for tab, cible, cols, comps in FORMULES:
        colonnes = ['NET', 'NET_N1'] if cols == 'NET' else cols
        for col in colonnes:
            declaree = _valeur(idx, tab, cible, col)
            recalc = _somme(idx, [(s, tab, rc, col) for s, rc in comps])
            if declaree is None or recalc is None: continue
            ecart = declaree - recalc
            resultats_formules.append({'type': 'agregat', 'tableau': tab, 'poste': cible, 'colonne': col, 'valeur_extraite': declaree, 'valeur_recalculee': recalc, 'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)})
    for tab, col_cible, comps in FORMULES_LIGNE:
        for (t, rc) in list(idx.keys()):
            if t != tab or rc == ' TOTAL ': continue
            declaree = _valeur(idx, tab, rc, col_cible)
            recalc = _somme(idx, [(s, tab, rc, c) for s, c in comps])
            if declaree is None or recalc is None: continue
            ecart = declaree - recalc
            resultats_formules.append({'type': 'ligne', 'tableau': tab, 'poste': rc, 'colonne': col_cible, 'valeur_extraite': declaree, 'valeur_recalculee': recalc, 'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)})
    for libelle, gravite, gauche, droite in CONTROLES:
        a, b = _somme(idx, gauche), _somme(idx, droite)
        if a is None or b is None:
            resultats_controles.append({'controle': libelle, 'gravite': gravite, 'statut': 'non_verifiable', 'valeur_a': a, 'valeur_b': b, 'ecart': None, 'commentaire': 'Un des deux membres est absent de l extraction.'})
            continue
        ecart = a - b
        resultats_controles.append({'controle': libelle, 'gravite': gravite, 'valeur_a': round(a, 2), 'valeur_b': round(b, 2), 'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)})
    ec_f = [f for f in resultats_formules if f['statut'] == 'ecart_significatif']
    ec_c = [c for c in resultats_controles if c['statut'] == 'ecart_significatif']
    critiques = [c for c in ec_c if c['gravite'] == 'critique']
    return {'tolerance_da': tolerance, 'formules': resultats_formules, 'controles_croises': resultats_controles,
            'synthese': {'formules_verifiees': len(resultats_formules), 'formules_en_ecart': len(ec_f),
                         'controles_verifies': len([c for c in resultats_controles if c['statut'] != 'non_verifiable']),
                         'controles_non_verifiables': len([c for c in resultats_controles if c['statut'] == 'non_verifiable']),
                         'controles_en_ecart': len(ec_c), 'ecarts_critiques': len(critiques),
                         'arbitrage_requis': bool(ec_f or ec_c), 'liasse_exploitable': not critiques}}
def enrichir_lignes(doc, rapport):
    index = {}
    for f in rapport['formules']: index[(f['tableau'], f['poste'], f['colonne'])] = f
    for page, tab, bloc in iter_blocs(doc):
        cible = page.setdefault('controles_donnees', {}).setdefault(tab, {})
        for cle, vals, _lib in lire_bloc(bloc):
            ctrl = {}
            for col, val in (vals or {}).items():
                if val is None or not isinstance(val, (int, float)) or isinstance(val, bool): continue
                f = index.get((tab, cle, col))
                if f: ctrl[col] = {'valeur_certaine': f['statut'] != 'ecart_significatif', 'valeur_extraite': f['valeur_extraite'], 'valeur_recalculee': f['valeur_recalculee'], 'ecart': f['ecart'], 'statut': f['statut']}
                else: ctrl[col] = {'valeur_certaine': None, 'statut': 'non_verifiable'}
            if ctrl: cible[cle] = ctrl
        if not cible: page['controles_donnees'].pop(tab, None)
    return doc
def appliquer_controles(doc):
    identite = construire_identite(doc)
    rapport = verifier_coherence(doc)
    enrichir_lignes(doc, rapport)
    doc['controles'] = rapport
    doc['schema_version'] = '13.0'
    s = rapport['synthese']
    doc['controles']['synthese']['identite_homogene'] = identite['dossier_homogene']
    doc['controles']['synthese']['arbitrage_requis'] = (s['arbitrage_requis'] or not identite['dossier_homogene'])
    return doc
print('✅ Moteur de coherence OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 15 — MAPPING EXCEL | BILANS_V13 (identique V12)
# ════════════════════════════════════════════════════════════
MAP_FEUILLES = {'ACTIF': '1. Bilan Actif', 'PASSIF': '2. Bilan Passif', 'TCR': '3. TCR', 'A1': '4. Stocks', 'A2': '4. Stocks', 'A3': '5. Charges & produits', 'A4': '5. Charges & produits', 'A5': '6. Amort. & Immo.', 'A6': '6. Amort. & Immo.', 'A7': '7. Cessions & Provisions', 'A8': '7. Cessions & Provisions', 'A81': '8. Pertes de valeurs', 'A82': '8. Pertes de valeurs', 'A9': '9. Résultat fiscal', 'A10': '10. Affectation & Particip.', 'A11': '10. Affectation & Particip.', 'A12': '11. Commissions & TAP', 'A13': '11. Commissions & TAP'}
MAP_COLONNES = {'ACTIF': {'montant_brut': 'B', 'amortissements_provisions_pertes': 'C', 'net_n': 'D', 'net_n1': 'E'}, 'PASSIF': {'n': 'B', 'n1': 'C'}, 'TCR': {'n_debit': 'B', 'n_credit': 'C', 'n1_debit': 'D', 'n1_credit': 'E'}, 'A1': {'solde_debut': 'B', 'debit': 'C', 'credit': 'D', 'solde_fin': 'E'}, 'A2': {'debit': 'B', 'credit': 'C', 'solde_debiteur': 'D', 'solde_crediteur': 'E'}, 'A3': {'montant': 'B'}, 'A4': {'montant': 'B'}, 'A5': {'dotations_cumulees_debut': 'B', 'dotations_exercice': 'C', 'diminutions_elements_sortis': 'D', 'dotations_cumulees_fin': 'E', 'dotations_fiscales_exercice': 'F', 'ecarts': 'G'}, 'A6': {'montants_bruts': 'B', 'tva_deduite': 'C', 'montant_net_a_amortir': 'D'}, 'A7': {'date_acquisition': 'B', 'montant_net_actif': 'C', 'amortissements_pratiques': 'D', 'valeur_nette_comptable': 'E', 'prix_cession': 'F', 'plus_value': 'G', 'moins_value': 'H'}, 'A8': {'provisions_cumulees_debut': 'B', 'dotations_exercice': 'C', 'reprises_exercice': 'D', 'provisions_cumulees_fin': 'E'}, 'A81': {'valeur_creance': 'B', 'perte_valeur_constituee': 'C'}, 'A82': {'valeur_nominale_debut': 'B', 'perte_valeur_constituee': 'C', 'valeur_nette_comptable': 'D'}, 'A9': {'montant': 'B'}, 'A10': {'montant': 'B'}, 'A11': {'capitaux_propres': 'B', 'dont_capital': 'C', 'quote_part_capital_pct': 'D', 'resultat_dernier_exercice': 'E', 'prets_avances': 'F', 'dividendes_encaisses': 'G', 'valeur_comptable_titres': 'H'}, 'A12': {'nif': 'B', 'adresse': 'C', 'montant_percu': 'D'}, 'A13': {'ca_imposable': 'B', 'ca_exonere': 'C', 'tap_acquittee': 'D'}}
MAP_LIGNES = {'ACTIF': {'ecarts_acquisition_goodwill': (11, False), 'immobilisations_incorporelles': (12, False), 'terrains': (14, False), 'batiments': (15, False), 'autres_immobilisations_corporelles': (16, False), 'immobilisations_en_concession': (17, False), 'immobilisations_en_cours': (18, False), 'titres_mis_en_equivalence': (20, False), 'autres_participations_creances': (21, False), 'autres_titres_immobilises': (22, False), 'prets_actifs_financiers_non_courants': (23, False), 'impots_differes_actif': (24, False), 'total_actif_non_courant': (25, True), 'stocks_encours': (27, False), 'clients': (29, False), 'autres_debiteurs': (30, False), 'impots_assimiles_actif': (31, False), 'autres_creances_assimiles': (32, False), 'placements_financiers_courants': (34, False), 'tresorerie_actif': (35, False), 'total_actif_courant': (36, True), 'total_general_actif': (37, True)},
 'PASSIF': {'capital_emis': (10, False), 'capital_non_appele': (11, False), 'primes_reserves': (12, False), 'ecart_reevaluation': (13, False), 'ecart_equivalence': (14, False), 'resultat_net_passif': (15, False), 'report_a_nouveau': (16, False), 'part_societe_consolidante': (17, False), 'part_minoritaires': (18, False), 'total_capitaux_propres': (19, True), 'emprunts_dettes_financieres': (21, False), 'impots_differes_provisionnes': (22, False), 'autres_dettes_non_courantes': (23, False), 'provisions_produits_avance': (24, False), 'total_passifs_non_courants': (25, True), 'fournisseurs_rattaches': (27, False), 'impots_passif': (28, False), 'autres_dettes': (29, False), 'tresorerie_passif': (30, False), 'total_passifs_courants': (31, True), 'total_general_passif': (32, True)},
 'TCR': {'ventes_marchandises': (10, False), 'produits_fabriques': (12, False), 'prestations_services': (13, False), 'ventes_travaux': (14, False), 'produits_annexes': (15, False), 'rabais_remises_ristournes_accordes': (16, False), 'chiffre_affaires_net': (17, True), 'production_stockee_destockee': (18, False), 'production_immobilisee': (19, False), 'subvention_exploitation': (20, False), 'production_exercice': (21, True), 'achats_marchandises_vendues': (22, False), 'matieres_premieres': (23, False), 'autres_approvisionnements': (24, False), 'variation_stocks': (25, False), 'rabais_remises_ristournes_obtenus_achats': (28, False),
  'achats_etudes_prestations': (26, False), 'autres_consommations': (27, False), 'sous_traitance_generale': (30, False), 'locations': (31, False), 'entretien_reparations': (32, False), 'primes_assurances': (33, False), 'personnel_exterieur': (34, False), 'remuneration_intermediaires': (35, False), 'publicite': (36, False), 'deplacements_missions': (37, False), 'rabais_remises_ristournes_obtenus_services': (39, False),
  'services_exterieurs': (29, True),
  'autres_services': (38, False), 'consommations_exercice': (40, True), 'valeur_ajoutee_exploitation': (41, True), 'charges_personnel': (42, False), 'impots_taxes_assimiles': (43, False), 'excedent_brut_exploitation': (44, True), 'autres_produits_operationnels': (45, False), 'autres_charges_operationnelles': (46, False), 'dotations_amortissements': (47, False), 'provisions': (48, False), 'pertes_valeur': (49, False), 'reprises_pertes_valeur_provisions': (50, False), 'resultat_operationnel': (51, True), 'produits_financiers': (52, False), 'charges_financieres': (53, False), 'resultat_financier': (54, True), 'resultat_ordinaire': (55, True), 'elements_extraordinaires_produits': (56, False), 'elements_extraordinaires_charges': (57, False), 'resultat_extraordinaire': (58, True), 'impots_exigibles_resultats': (59, False), 'impots_differes_resultats': (60, False), 'resultat_net_exercice': (61, True)},
 'A1': {'stocks_marchandises': (11, False), 'matieres_fournitures': (12, False), 'autres_approvisionnements': (13, False), 'encours_production_biens': (14, False), 'encours_production_services': (15, False), 'stocks_produits': (16, False), 'stocks_provenant_immobilisations': (17, False), 'stocks_exterieur': (18, False), 'total': (19, True)},
 'A3': {'charges_locatives': (11, False), 'etudes_recherches': (12, False), 'documentation_divers': (13, False), 'transports_biens': (14, False), 'frais_postaux': (15, False), 'services_bancaires': (16, False), 'cotisations_divers': (17, False), 'total_autres_services': (18, True), 'remunerations_personnel': (20, False), 'remuneration_exploitant': (21, False), 'cotisations_sociales': (22, False), 'charges_sociales_exploitant': (23, False), 'autres_charges_sociales': (24, False), 'autres_charges_personnel': (25, False), 'total_charges_personnel': (26, True), 'impots_sur_remunerations': (28, False), 'impots_non_recuperables': (29, False), 'autres_impots_taxes': (30, False), 'total_impots': (31, True), 'total_general': (32, True)},
 'A4': {'redevances_concessions_charges': (37, False), 'moins_values_sorties_actifs': (38, False), 'jetons_presence_charges': (39, False), 'pertes_creances_irrecouvrables': (40, False), 'quote_part_operations_commun_charges': (41, False), 'amendes_penalites_dons': (42, False), 'charges_exceptionnelles_gestion': (43, False), 'autres_charges_gestion': (44, False), 'total_charges': (45, True), 'redevances_concessions_produits': (50, False), 'plus_values_sorties_actifs': (51, False), 'jetons_presence_produits': (52, False), 'quotes_parts_subventions_virees': (53, False), 'quote_part_operations_commun_produits': (54, False), 'rentrees_creances_amorties': (55, False), 'produits_exceptionnels_gestion': (56, False), 'autres_produits_gestion': (57, False), 'total_produits': (58, True)},
 'A5': {'goodwill': (10, False), 'immobilisations_incorporelles': (11, False), 'immobilisations_corporelles': (12, False), 'participations': (13, False), 'autres_actifs_financiers_non_courants': (14, False), 'total': (15, True)},
 'A6': {'goodwill': (20, False), 'immobilisations_incorporelles': (21, False), 'immobilisations_corporelles': (22, False), 'participations': (23, False), 'autres_actifs_financiers_non_courants': (24, False), 'total': (25, True)},
 'A8': {'pertes_valeur_stocks': (26, False), 'pertes_valeur_creances': (27, False), 'pertes_valeur_actions': (28, False), 'provisions_pensions': (29, False), 'provisions_litiges': (30, False), 'autres_provisions_personnel': (31, False), 'provisions_impots': (32, False), 'autres_provisions': (33, False), 'total': (34, True)},
 'A9': {'resultat_net_benefice': (9, False), 'resultat_net_perte': (10, False), 'charges_immeubles_non_affectes': (12, False), 'quote_part_cadeaux_publicitaires': (13, False), 'quote_part_sponsoring': (14, False), 'frais_reception': (15, False), 'cotisations_dons': (16, False), 'impots_taxes_non_deductibles': (17, False), 'provisions_non_deductibles': (18, False), 'amortissements_non_deductibles': (19, False), 'quote_part_frais_rd': (20, False), 'amortissements_credit_bail_preneur': (21, False), 'loyers_hors_produits_financiers_bailleur': (22, False), 'ibs_impot_exigible': (23, False), 'ibs_impot_differe': (24, False), 'pertes_valeur_non_deductibles': (25, False), 'amendes_penalites': (26, False), 'autres_reintegrations': (27, False), 'total_reintegrations': (28, True), 'plus_values_cession_actif_immobilise': (30, False), 'produits_plus_values_actions_bourse': (31, False), 'revenus_distribution_benefices': (32, False), 'amortissements_credit_bail_bailleur': (33, False), 'loyers_hors_charges_financieres_preneur': (34, False), 'complement_amortissements': (35, False), 'autres_deductions': (36, False), 'total_deductions': (37, True), 'total_deficits_a_deduire': (43, True), 'resultat_fiscal_benefice': (44, True), 'resultat_fiscal_deficit': (45, True)},
 'A10': {'origine_report_a_nouveau_n1': (11, False), 'origine_resultat_n1': (12, False), 'origine_prelevements_reserves': (13, False), 'origine_total': (14, True), 'affectation_reserves': (16, False), 'affectation_augmentation_capital': (17, False), 'affectation_dividendes': (18, False), 'affectation_report_a_nouveau': (19, False), 'affectation_total': (20, True)}}
MAP_DYNAMIQUES = {'A2': (25, 32, None), 'A7': (11, 20, 'A'), 'A81': (10, 23, 'A'), 'A82': (29, 42, 'A'), 'A11': (26, 34, 'A'), 'A12': (10, 24, 'A'), 'A13': (30, 41, 'A')}
MAP_ENTETE = {'nif': 'B4', 'entreprise': 'B5', 'exercice': 'B6'}
MAP_MILLESIMES = [('1. Bilan Actif', 'B8', 'N'), ('1. Bilan Actif', 'E8', 'N1'), ('2. Bilan Passif', 'B8', 'N'), ('2. Bilan Passif', 'C8', 'N1'), ('3. TCR', 'B8', 'N'), ('3. TCR', 'D8', 'N1')]
print('✅ Mapping Excel OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 16 — TRANSPOSITION V13
# Nouveautes V13 :
#  - modele + recalc resolus depuis /mnt/Risk/Model
#  - ecarts d'arrondi (+/- 1 DA) : commentaire + surbrillance jaune (pas rouge)
#  - tracabilite : en-tete de chaque feuille = PDF source + page(s) scannee(s)
# ════════════════════════════════════════════════════════════
MODEL_DIR = Path('/mnt/Risk/Model')
_cand = []
for pat in ('Liasse*.xlsx', 'liasse*.xlsx', 'liaisse*.xlsx'):
    for p in sorted(MODEL_DIR.glob(pat)):
        if p not in _cand: _cand.append(p)
MODELE_XLSX = _cand[0] if _cand else MODEL_DIR / 'Liasse_fiscale_G2_BNP_Paribas_El_Djazair.xlsx'
RECALC_TOOL = MODEL_DIR / 'recalc_liasse.py'
RECALC_DISPONIBLE = RECALC_TOOL.exists() and bool(shutil.which('soffice'))
_ROUGE = Border([Side(style='medium', color='FF0000')] * 4)
_ROUGE_FOND = PatternFill('solid', start_color='FFC7CE')
_ROUGE_TEXTE = Font(bold=True, color='9C0006')
_JAUNE_FOND = PatternFill('solid', start_color='FFF2CC')
_LIBRE = Protection(locked=False)
_VERROU = Protection(locked=True)
def _est_formule(c):
    return isinstance(c.value, str) and c.value.startswith('=')
def _postes_du_document(doc):
    fixes, dynamiques = [], {}
    for _page, tab, bloc in iter_blocs(doc):
        for cle, vals, libelle in lire_bloc(bloc):
            vals = vals or {}
            est_libre = str(cle).startswith('ligne_')
            if not est_libre and tab in MAP_LIGNES and cle in MAP_LIGNES[tab]:
                for col, v in vals.items():
                    if v is not None: fixes.append((tab, cle, col, v))
            elif tab in MAP_DYNAMIQUES:
                if any(v is not None for v in vals.values()) or libelle: dynamiques.setdefault(tab, []).append({'libelle': libelle, 'valeurs': vals})
            else:
                for col, v in vals.items():
                    if v is not None: fixes.append((tab, cle, col, v))
    return fixes, dynamiques
def _ecrire_valeurs(wb, doc):
    fixes, dynamiques = _postes_du_document(doc)
    ecrits, totaux, ignores, par_tableau = 0, [], [], {}
    ident = doc.get('identite') or {}
    ws1 = wb[MAP_FEUILLES['ACTIF']]
    if ident.get('nif_15'): ws1[MAP_ENTETE['nif']] = ident['nif_15']
    if ident.get('entreprise'): ws1[MAP_ENTETE['entreprise']] = ident['entreprise']
    if ident.get('exercice_clos'): ws1[MAP_ENTETE['exercice']] = ident['exercice_clos']
    for feuille, coord, quel in MAP_MILLESIMES:
        annee = ident.get('annee_n') if quel == 'N' else ident.get('annee_n1')
        if annee: wb[feuille][coord] = ('N : ' + str(annee)) if quel == 'N' else ('N-1 : ' + str(annee))
    for tab, rc, col, val in fixes:
        ligne_info = MAP_LIGNES.get(tab, {}).get(rc); colonne = MAP_COLONNES.get(tab, {}).get(col)
        if not ligne_info or not colonne: ignores.append((tab, rc, col)); continue
        ligne, _ = ligne_info
        cible = wb[MAP_FEUILLES[tab]][colonne + str(ligne)]
        if _est_formule(cible):
            if isinstance(val, (int, float)) and not isinstance(val, bool): totaux.append((MAP_FEUILLES[tab], colonne + str(ligne), tab, rc, col, float(val)))
            continue
        cible.value = val; ecrits += 1; par_tableau[tab] = par_tableau.get(tab, 0) + 1
    for tab, lignes in dynamiques.items():
        debut, fin, col_lib = MAP_DYNAMIQUES[tab]; ws = wb[MAP_FEUILLES[tab]]; r = debut
        for item in lignes:
            if r > fin: ignores.append((tab, 'ligne_' + str(r), 'depassement_capacite')); break
            if col_lib and item.get('libelle'): ws[col_lib + str(r)] = item['libelle']
            for col, v in (item.get('valeurs') or {}).items():
                colonne = MAP_COLONNES.get(tab, {}).get(col)
                if colonne and v is not None:
                    cc = ws[colonne + str(r)]
                    if _est_formule(cc):
                        if isinstance(v, (int, float)) and not isinstance(v, bool): totaux.append((MAP_FEUILLES[tab], colonne + str(r), tab, 'ligne_' + str(r), col, float(v)))
                    else: cc.value = v; ecrits += 1; par_tableau[tab] = par_tableau.get(tab, 0) + 1
            r += 1
    return ecrits, totaux, ignores, par_tableau
def _ecrire_trace(wb, doc):
    pdf_name = (doc.get('document') or {}).get('fichier_source') or '?'
    sources = {}
    for page in doc.get('pages', []):
        if (page.get('classification') or {}).get('page_blanche'): continue
        num = page.get('numero_page_scannee')
        for tab in (page.get('donnees') or {}):
            if tab in ('brut', 'AUTRE'): continue
            f = MAP_FEUILLES.get(tab)
            if f and f in wb.sheetnames: sources.setdefault(f, set()).add(num)
    for feuille, nums in sources.items():
        ws = wb[feuille]
        for col in ('H', 'J', 'L'):
            c1, c2 = ws[col + '4'], ws[col + '5']
            if not _est_formule(c1) and c1.value in (None, '') and not _est_formule(c2) and c2.value in (None, ''):
                c1.value = 'Source : ' + str(pdf_name)
                c2.value = 'Page(s) scannee(s) : ' + ' / '.join(str(n) for n in sorted(nums))
                for c in (c1, c2): c.font = Font(italic=True, size=8, color='808080')
                break
def _comparer_et_annoter(chemin_calcule, wb, totaux, doc=None, tolerance=TOLERANCE_DA):
    calc = None
    if chemin_calcule is not None:
        try: calc = openpyxl.load_workbook(chemin_calcule, data_only=True)
        except Exception: calc = None
    replis = {}
    if calc is None and doc is not None:
        for f in (doc.get('controles') or {}).get('formules', []):
            replis[(f['tableau'], f['poste'], f['colonne'])] = f['valeur_recalculee']
            # ── Dupliquer les pseudo-colonnes TCR vers les colonnes Excel reelles ──
            # NET  = credit_N - debit_N   → on reporte aussi sur n_credit / n_debit
            # NET_N1 = credit_N1 - debit_N1 → idem pour n1_credit / n1_debit
            if f['tableau'] == 'TCR' and f['colonne'] == 'NET':
                replis[('TCR', f['poste'], 'n_credit')] = f['valeur_recalculee']
            elif f['tableau'] == 'TCR' and f['colonne'] == 'NET_N1':
                replis[('TCR', f['poste'], 'n1_credit')] = f['valeur_recalculee']
    anomalies, arrondis = [], []
    for feuille, coord, tab, rc, col, extraite in totaux:
        if calc is not None:
            recalc = calc[feuille][coord].value
            if recalc is None: continue
            recalc = float(recalc)
        else:
            if (tab, rc, col) not in replis: continue
            recalc = float(replis[(tab, rc, col)])
        ecart = extraite - recalc
        if abs(ecart) <= 1e-9: continue
        c = wb[feuille][coord]
        if abs(ecart) <= tolerance:
            c.comment = Comment('ECART D ARRONDI (+/- 1 DA)\n--------------------\nValeur lue sur la liasse : ' + format(extraite, ',.2f') + '\nValeur recalculee        : ' + format(recalc, ',.2f') + '\nEcart                    : ' + format(ecart, ',.2f') + '\n\nSimple arrondi : aucune correction necessaire.', 'Controle extraction')
            c.comment.width = 300; c.comment.height = 150
            c.fill = _JAUNE_FOND
            arrondis.append({'tableau': tab, 'poste': rc, 'colonne': col, 'feuille': feuille, 'cellule': coord, 'valeur_extraite': extraite, 'valeur_recalculee': recalc, 'ecart': round(ecart, 2)})
            continue
        c.border = _ROUGE; c.fill = _ROUGE_FOND; c.font = _ROUGE_TEXTE
        c.comment = Comment('INCOHERENCE DETECTEE\n--------------------\nValeur lue sur la liasse : ' + format(extraite, ',.2f') + '\nValeur recalculee        : ' + format(recalc, ',.2f') + '\nEcart                    : ' + format(ecart, ',.2f') + '\n\nLa cellule affiche la valeur RECALCULEE.', 'Controle extraction')
        c.comment.width = 340; c.comment.height = 190
        anomalies.append({'tableau': tab, 'poste': rc, 'colonne': col, 'feuille': feuille, 'cellule': coord, 'valeur_extraite': extraite, 'valeur_recalculee': recalc, 'ecart': round(ecart, 2)})
    return anomalies, arrondis

def _annoter_ecarts_python(wb, doc):
    """Annote les ecarts detectes par le moteur Python (cellule 15) qui ne sont
    pas couverts par la comparaison cellule par cellule.

    Cas principal : les colonnes NET et NET_N1 du TCR, qui sont des pseudo-colonnes
    (credit - debit) sans cellule Excel correspondante. L ecart est annote sur la
    cellule CREDIT de la ligne concernee (colonne C pour N, E pour N-1).
    """
    rapport = (doc.get('controles') or {}).get('formules', [])
    if not rapport:
        return 0

    # Mapping colonne pseudo -> colonne Excel d annotation
    PSEUDO_COL_MAP = {
        'NET':    'C',     # credit N
        'NET_N1': 'E',     # credit N-1
    }

    n_annotes = 0
    for f in rapport:
        if f['statut'] not in ('ecart_significatif', 'coherent_arrondi'):
            continue
        tab = f['tableau']
        poste = f['poste']
        col = f['colonne']

        # Ne traiter que les pseudo-colonnes
        if col not in PSEUDO_COL_MAP:
            continue

        # Trouver la ligne Excel
        feuille = MAP_FEUILLES.get(tab)
        if not feuille or tab not in MAP_LIGNES:
            continue
        info = MAP_LIGNES[tab].get(poste)
        if not info:
            continue
        ligne, _calculee = info

        col_xl = PSEUDO_COL_MAP[col]
        coord = col_xl + str(ligne)
        ws = wb[feuille]
        c = ws[coord]

        ecart = f['ecart']
        extraite = f['valeur_extraite']
        recalculee = f['valeur_recalculee']
        statut = f['statut']

        if statut == 'ecart_significatif':
            c.border = _ROUGE
            c.fill = _ROUGE_FOND
            c.font = _ROUGE_TEXTE
        # Pour coherent_arrondi : pas de rouge, juste un commentaire informatif

        commentaire = (
            'CONTROLE ' + statut.upper().replace('_', ' ') + '\n'
            '--------------------\n'
            'Solde net extrait (credit-debit) : ' + format(extraite, ',.2f') + '\n'
            'Solde net recalcule              : ' + format(recalculee, ',.2f') + '\n'
            'Ecart                            : ' + format(ecart, ',.2f') + '\n\n'
        )
        if statut == 'ecart_significatif':
            commentaire += ('La valeur recalculee par formule differe de la valeur\n'
                           'extraite du document. Verifier la page source.')
        else:
            commentaire += ('Ecart d arrondi (1 DA) — pas d action requise.\n'
                           'Le total recalcule par formule est quasi identique.')

        c.comment = Comment(commentaire, 'Controle coherence')
        c.comment.width = 340
        c.comment.height = 170
        n_annotes += 1

    return n_annotes


def _feuille_archive(wb, doc, anomalies, arrondis, ignores):
    if '0. Données extraites' in wb.sheetnames: del wb['0. Données extraites']
    ws = wb.create_sheet('0. Données extraites', 0)
    ident = doc.get('identite') or {}; synth = (doc.get('controles') or {}).get('synthese', {})
    ws['A1'] = 'DONNEES EXTRAITES DE LA LIASSE — PIECE DE REFERENCE'; ws['A1'].font = Font(bold=True, size=14)
    ws['A2'] = 'Valeurs telles que lues sur la liasse deposee. Les feuilles suivantes sont une copie de travail dont les totaux sont recalcules.'; ws['A2'].font = Font(italic=True, size=9)
    infos = [('Fichier source', doc.get('document', {}).get('fichier_source')), ('Date extraction', doc.get('document', {}).get('date_extraction')), ('Entreprise', ident.get('entreprise')), ('NIF (15 positions)', ident.get('nif_15')), ('Exercice clos', ident.get('exercice_clos')), ('Exercice N', ident.get('annee_n')), ('Exercice N-1', ident.get('annee_n1')), ('Dossier homogene', 'OUI' if ident.get('dossier_homogene') else 'NON'), ('Formules en ecart', synth.get('formules_en_ecart')), ('Controles en ecart', synth.get('controles_en_ecart')), ('Ecarts critiques', synth.get('ecarts_critiques')), ('Arbitrage requis', 'OUI' if synth.get('arbitrage_requis') else 'NON')]
    r = 4
    for lib, val in infos: ws.cell(r, 1, lib).font = Font(bold=True); ws.cell(r, 2, val); r += 1
    r += 1; ws.cell(r, 1, 'ALERTES D IDENTITE').font = Font(bold=True, size=12); r += 1
    for a in ident.get('alertes', []): ws.cell(r, 1, a['gravite'].upper()).font = Font(bold=True, color='9C0006'); ws.cell(r, 2, a['message']); r += 1
    if not ident.get('alertes'): ws.cell(r, 2, 'Aucune — toutes les pages designent le meme dossier.'); r += 1
    r += 1; ws.cell(r, 1, 'ECARTS SIGNIFICATIFS (DECLARE / RECALCULE)').font = Font(bold=True, size=12); r += 1
    for i, h in enumerate(['Tableau', 'Poste', 'Colonne', 'Cellule', 'Valeur extraite', 'Valeur recalculee', 'Ecart'], 1):
        c = ws.cell(r, i, h); c.font = Font(bold=True, color='FFFFFF'); c.fill = PatternFill('solid', start_color='1F4E79')
    r += 1
    for a in anomalies:
        ws.cell(r, 1, a['tableau']); ws.cell(r, 2, a['poste']); ws.cell(r, 3, a['colonne']); ws.cell(r, 4, a['feuille'] + '!' + a['cellule'])
        for i, cle in enumerate(['valeur_extraite', 'valeur_recalculee', 'ecart'], 5): ws.cell(r, i, a[cle]).number_format = '#,##0.00'
        r += 1
    if not anomalies: ws.cell(r, 2, 'Aucun ecart significatif.'); r += 1
    r += 1; ws.cell(r, 1, 'ECARTS D ARRONDI (+/- 1 DA) — COMMENTES DANS LES FEUILLES').font = Font(bold=True, size=12); r += 1
    for a in arrondis: ws.cell(r, 1, a['tableau']); ws.cell(r, 2, a['poste']); ws.cell(r, 3, a['feuille'] + '!' + a['cellule']); ws.cell(r, 4, a['ecart']); r += 1
    if not arrondis: ws.cell(r, 2, 'Aucun ecart d arrondi.'); r += 1
    if ignores:
        r += 1; ws.cell(r, 1, 'POSTES NON REPORTES').font = Font(bold=True, size=12); r += 1
        for tab, rc, col in ignores: ws.cell(r, 1, tab); ws.cell(r, 2, rc); ws.cell(r, 3, col); r += 1
    for col, w in zip('ABCDEFG', [26, 46, 20, 24, 20, 20, 16]): ws.column_dimensions[col].width = w
    return ws
def _proteger_formules(wb):
    n_libres, n_verrous = 0, 0
    for ws in wb.worksheets:
        for ligne in ws.iter_rows():
            for c in ligne:
                if isinstance(c.value, str) and c.value.startswith('='): c.protection = _VERROU; n_verrous += 1
                else: c.protection = _LIBRE; n_libres += 1
        ws.protection.sheet = True
    return n_libres, n_verrous
def transposer(doc, sortie_xlsx, modele=None, tolerance=TOLERANCE_DA):
    modele = Path(modele or MODELE_XLSX); sortie = Path(sortie_xlsx); tmp = sortie.with_suffix('.tmp.xlsx')
    shutil.copy(modele, tmp)
    wb = openpyxl.load_workbook(tmp)
    for ws in wb.worksheets: ws.protection.sheet = False
    ecrits, totaux, ignores, par_tableau = _ecrire_valeurs(wb, doc)
    _ecrire_trace(wb, doc)
    wb.save(tmp)
    recalcule = False
    if RECALC_DISPONIBLE:
        try:
            subprocess.run([sys.executable, str(RECALC_TOOL), str(tmp), '180'], capture_output=True, timeout=300); recalcule = True
        except Exception as e: log('Recalcul LibreOffice indisponible : ' + str(e))
    else: log('LibreOffice absent — detection des ecarts par le moteur Python.')
    wb = openpyxl.load_workbook(tmp)
    anomalies, arrondis = _comparer_et_annoter(tmp if recalcule else None, wb, totaux, doc, tolerance)
    n_annotes_python = _annoter_ecarts_python(wb, doc)
    _feuille_archive(wb, doc, anomalies, arrondis, ignores)
    # ── En-tete par feuille : entreprise, NIF, exercice, pages sources ──
    ident = doc.get('identite') or {}
    pages_par_feuille = {}
    for _page, tab, _bloc in iter_blocs(doc):
        f = MAP_FEUILLES.get(tab)
        if f:
            info = _page.get('entete_page') or {}
            pages_par_feuille.setdefault(f, []).append({
                'num': _page.get('numero_page_scannee'),
                'pid': _page.get('page_id'),
            })

    feuilles_renseignees = set(pages_par_feuille.keys())
    for ws in wb.worksheets:
        if ws.title.startswith('0.') or ws.title.startswith('12.'):
            continue
        # En-tete : renvoi vers la feuille 1 (deja en place via MAP_ENTETE)
        # Ajouter le numero de page source et le document
        if ws.title in pages_par_feuille:
            nums = sorted(set(p['num'] for p in pages_par_feuille[ws.title]
                              if p['num'] is not None))
            src_txt = ', '.join(str(n) for n in nums)
        else:
            src_txt = ''
        # Ecrire dans une cellule libre en haut a droite (colonne G)
        ws['G3'] = 'Pages PDF : ' + src_txt if src_txt else ''
        ws['G4'] = str(doc.get('document', {}).get('fichier_source') or '')
        from openpyxl.styles import Font as _F
        ws['G3'].font = _F(name='Calibri', italic=True, size=8, color='808080')
        ws['G4'].font = _F(name='Calibri', italic=True, size=8, color='808080')

    # ── Cacher les feuilles sans données extraites ──
    for ws in wb.worksheets:
        if ws.title.startswith('0.') or ws.title.startswith('12.'):
            continue
        if ws.title == MAP_FEUILLES.get('ACTIF'):
            continue  # Ne jamais cacher la feuille principale
        if ws.title not in feuilles_renseignees:
            ws.sheet_state = 'hidden'

    n_libres, n_verrous = _proteger_formules(wb)
    wb.save(sortie); tmp.unlink(missing_ok=True)
    if RECALC_DISPONIBLE:
        try: subprocess.run([sys.executable, str(RECALC_TOOL), str(sortie), '180'], capture_output=True, timeout=300)
        except Exception as e: log('Recalcul final indisponible : ' + str(e))
    doc.setdefault('controles', {})['transposition'] = {'classeur': sortie.name, 'valeurs_ecrites': ecrits, 'totaux_compares': len(totaux), 'ecarts_signales': len(anomalies), 'ecarts_arrondis': len(arrondis), 'postes_non_reportes': len(ignores), 'cellules_verrouillees': n_verrous, 'cellules_saisissables': n_libres, 'recalcul_libreoffice': RECALC_DISPONIBLE, 'valeurs_par_tableau': par_tableau, 'detail_ecarts': anomalies, 'detail_arrondis': arrondis, 'ecarts_annotes_controles_python': n_annotes_python}
    return doc['controles']['transposition']
print('✅ Transposition V13 OK | modele: ' + str(MODELE_XLSX) + ' | recalc: ' + str(RECALC_TOOL))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 17 — EXECUTION POST-EXTRACTION | BILANS_V13
# ════════════════════════════════════════════════════════════
CONTROLE_DIR = OUTPUT_DIR / 'json_controles'
XLSX_DIR = OUTPUT_DIR / 'liasses_xlsx'
CONTROLE_DIR.mkdir(parents=True, exist_ok=True)
XLSX_DIR.mkdir(parents=True, exist_ok=True)
def traiter_dossier(chemin_json, avec_excel=True):
    doc = json.loads(Path(chemin_json).read_text(encoding='utf-8'))
    doc = appliquer_controles(doc)
    stem = Path(chemin_json).stem
    if avec_excel and Path(MODELE_XLSX).exists():
        try: transposer(doc, XLSX_DIR / (stem + '.xlsx'))
        except Exception as e: log('❌ Transposition ' + stem + ' : ' + str(e))
    elif avec_excel: log('⚠️ Modele Excel introuvable : ' + str(MODELE_XLSX))
    sortie = CONTROLE_DIR / (stem + '.controle.json')
    sortie.write_text(json.dumps(doc, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    return doc, sortie
def resumer(doc):
    ident = doc.get('identite') or {}; s = (doc.get('controles') or {}).get('synthese', {}); t = (doc.get('controles') or {}).get('transposition', {})
    print('  ' + str(ident.get('entreprise') or '(entreprise inconnue)') + '  |  NIF ' + str(ident.get('nif_15') or '—') + '  |  ' + str(ident.get('libelle_n') or 'N'))
    print('    Dossier homogene   : ' + ('OUI' if ident.get('dossier_homogene') else 'NON'))
    print('    Formules verifiees : ' + str(s.get('formules_verifiees')) + '  (ecarts : ' + str(s.get('formules_en_ecart')) + ')')
    print('    Controles croises  : ' + str(s.get('controles_verifies')) + '  (ecarts : ' + str(s.get('controles_en_ecart')) + ', non verifiables : ' + str(s.get('controles_non_verifiables')) + ')')
    print('    Ecarts critiques   : ' + str(s.get('ecarts_critiques')))
    print('    Liasse exploitable : ' + ('OUI' if s.get('liasse_exploitable') else 'NON'))
    if t:
        print('    Excel : ' + str(t.get('valeurs_ecrites')) + ' valeurs | ' + str(t.get('ecarts_signales')) + ' ecart(s) significatif(s) | ' + str(t.get('ecarts_arrondis')) + ' arrondi(s) commente(s) | ' + str(t.get('cellules_verrouillees')) + ' verrous | controles annotes=' + str(t.get('ecarts_annotes_controles_python', 0)) + ' | cellules verrouillees')
        detail = t.get('valeurs_par_tableau') or {}
        if detail: print('      par tableau : ' + ', '.join(k + '=' + str(v) for k, v in sorted(detail.items())))
        if not t.get('valeurs_ecrites'): print('      ⚠️ AUCUNE valeur reportee — verifier mapping/structure.')
    for a in (t.get('detail_arrondis') or []): print('      ARRONDI ' + a['feuille'] + '!' + a['cellule'] + ' : ' + str(a['valeur_extraite']) + ' vs ' + str(a['valeur_recalculee']) + ' (ecart ' + str(a['ecart']) + ')')
    for c in (doc.get('controles') or {}).get('controles_croises', []):
        if c['statut'] == 'ecart_significatif': print('      ECART [' + c['gravite'] + '] ' + c['controle'] + ' : ' + format(c['valeur_a'], ',.0f') + ' vs ' + format(c['valeur_b'], ',.0f'))
fichiers = sorted(JSON_DIR.glob('*.json'))
log('Post-traitement V13 de ' + str(len(fichiers)) + ' dossier(s)')
for f in fichiers:
    try:
        doc, sortie = traiter_dossier(f)
        print(); print('📄 ' + f.name); resumer(doc)
    except Exception as e: log('❌ ' + f.name + ' : ' + str(e))
print(); print('✅ Post-traitement V13 termine'); print('   JSON controles : ' + str(CONTROLE_DIR)); print('   Classeurs      : ' + str(XLSX_DIR))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 18 — TESTS HORS MODELE | BILANS_V13
# ════════════════════════════════════════════════════════════
_ok, _tot = 0, 0
def _t(libelle, obtenu, attendu):
    global _ok, _tot
    _tot += 1; bon = obtenu == attendu; _ok += bon
    print(('✅' if bon else '❌') + ' ' + libelle + ('' if bon else '  obtenu=' + repr(obtenu) + '  attendu=' + repr(attendu)))
print('--- NIF 15 ---')
_t('conforme', nif_15('002119116225582'), ('002119116225582', 'CONFORME'))
_t('court complete', nif_15('2119116225582'), ('002119116225582', 'COMPLETE'))
_t('absent', nif_15(None), (None, 'ABSENT'))
print('--- Millesimes ---')
_t('annee exercice', annee_exercice({'exercice': '31/12/2025'}), 2025)
print('--- Coherence : bilan equilibre ---')
_CA = ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1']; _CP = ['n', 'n1']
def _lignes(rows, cols): return {'lignes': [{'row_code': rc, 'valeurs': dict(zip(cols, v))} for rc, v in rows.items()]}
_doc3 = {'pages': [
 {'classification': {'types': ['ACTIF'], 'page_blanche': False}, 'entete_page': {}, 'donnees': {'ACTIF': _lignes({'immobilisations_incorporelles': [300000, 127500, 172500, 232500], 'stocks_encours': [53035400, None, 53035400, 257185175], 'clients': [69060109, None, 69060109, 44591141], 'total_actif_courant': [122095509, 0, 122095509, 703096720], 'total_general_actif': [122395509, 127500, 122268009, 703329220]}, _CA)}},
 {'classification': {'types': ['PASSIF'], 'page_blanche': False}, 'entete_page': {}, 'donnees': {'PASSIF': _lignes({'capital_emis': [205000000, 205000000], 'primes_reserves': [110191620, 44695277], 'resultat_net_passif': [29275402, 65496343], 'total_capitaux_propres': [344467022, 315191620], 'total_passifs_non_courants': [0, 0], 'fournisseurs_rattaches': [589759, 359932], 'autres_dettes': [401882646, 139822555], 'total_passifs_courants': [402472405, 140182487], 'total_general_passif': [746939427, 455374107]}, _CP)}}]}
_rap = verifier_coherence(_doc3)
_t('aucun ecart critique', _rap['synthese']['ecarts_critiques'], 0)
print('--- Arrondi : commentaire attendu ---')
_doc4 = json.loads(json.dumps(_doc3))
for _l in _doc4['pages'][0]['donnees']['ACTIF']['lignes']:
    if _l['row_code'] == 'total_actif_courant': _l['valeurs']['net_n'] = 122095510  # +1 : arrondi
_rap4 = verifier_coherence(_doc4)
_arr = [f for f in _rap4['formules'] if f['statut'] == 'coherent_arrondi']
_t('ecart +1 detecte comme arrondi', len(_arr) >= 1, True)
print(); print(str(_ok) + '/' + str(_tot) + ' tests passes')